In [1]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 1 - CONFIGURAÇÃO INICIAL
# ============================================================

!pip -q install openpyxl pandas

import os
import re
import json
import unicodedata
import pandas as pd

from collections import defaultdict
from google.colab import files
from openpyxl import load_workbook

print("=" * 70)
print("APRESENTADOR 360 V2")
print("=" * 70)
print("Configuração carregada com sucesso.")

APRESENTADOR 360 V2
Configuração carregada com sucesso.


In [2]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 2 - UPLOAD DA MATRIZ E ESTADO GLOBAL
# ============================================================

from google.colab import files
from openpyxl import load_workbook

# ------------------------------------------------------------
# ESTADO CENTRAL DO PROJETO
# ------------------------------------------------------------

estado = {
    "arquivo_matriz": None,
    "abas": [],
    "operadoras": [],
    "cenarios": {},
    "empresa": {},
    "elegibilidade": {},
    "faixa_etaria": {}
}

print("=" * 70)
print("UPLOAD DA MATRIZ")
print("=" * 70)

uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("Nenhum arquivo enviado.")

estado["arquivo_matriz"] = list(uploaded.keys())[0]

print("\nArquivo carregado:")
print(estado["arquivo_matriz"])

# ------------------------------------------------------------
# LER ABAS VISÍVEIS
# ------------------------------------------------------------

wb = load_workbook(
    estado["arquivo_matriz"],
    data_only=True
)

print("\n" + "=" * 70)
print("ABAS VISÍVEIS ENCONTRADAS")
print("=" * 70)

for ws in wb.worksheets:

    if ws.sheet_state == "visible":

        estado["abas"].append(ws.title)

        print(f"✅ {ws.title}")

wb.close()

print("\n" + "=" * 70)
print(f"TOTAL DE ABAS VISÍVEIS: {len(estado['abas'])}")
print("=" * 70)

print("\nEstado inicial criado com sucesso.")

UPLOAD DA MATRIZ


Saving Mezzo_Matriz PME_2026_08_v2.xlsx to Mezzo_Matriz PME_2026_08_v2.xlsx

Arquivo carregado:
Mezzo_Matriz PME_2026_08_v2.xlsx

ABAS VISÍVEIS ENCONTRADAS
✅ Base
✅ Dinamica
✅ CC
✅ SC

TOTAL DE ABAS VISÍVEIS: 4

Estado inicial criado com sucesso.


In [3]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 3 - IDENTIFICAR ABAS DE COTAÇÃO
# ============================================================

from openpyxl import load_workbook

print("=" * 70)
print("IDENTIFICANDO ABAS COM CENÁRIOS")
print("=" * 70)

estado["abas_cotacao"] = []

PALAVRAS_CHAVE = [
    "OPERADORA",
    "PLANO",
    "FAIXA ETÁRIA",
    "VIDAS",
    "PER CAPITA",
    "TOTAL / FATURA"
]

wb = load_workbook(
    estado["arquivo_matriz"],
    data_only=True
)

for ws in wb.worksheets:

    # Ignora abas ocultas
    if ws.sheet_state != "visible":
        continue

    score = 0

    for linha in ws.iter_rows():

        for celula in linha:

            if celula.value is None:
                continue

            texto = str(celula.value).upper().strip()

            if texto in PALAVRAS_CHAVE:
                score += 1

    if score >= 3:

        estado["abas_cotacao"].append({
            "aba": ws.title,
            "score": score
        })

wb.close()

print("\nABAS IDENTIFICADAS COMO COTAÇÃO:\n")

for item in estado["abas_cotacao"]:

    print(
        f"✅ {item['aba']} "
        f"(score: {item['score']})"
    )

print("\nTotal encontradas:",
      len(estado["abas_cotacao"]))

IDENTIFICANDO ABAS COM CENÁRIOS

ABAS IDENTIFICADAS COMO COTAÇÃO:

✅ CC (score: 123)
✅ SC (score: 123)

Total encontradas: 2


In [4]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 4 - MAPEAR OPERADORAS E CENÁRIOS AUTOMATICAMENTE
# ============================================================

from openpyxl import load_workbook
from collections import OrderedDict

print("=" * 70)
print("MAPEANDO OPERADORAS E CENÁRIOS")
print("=" * 70)

# ------------------------------------------------------------
# FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def texto_limpo(valor):
    if valor is None:
        return ""

    return " ".join(
        str(valor).replace("\n", " ").split()
    ).strip()


def candidato_valido_operadora(valor):
    """
    Valida o texto encontrado ao lado de OPERADORA.

    Não existe lista fixa de nomes de operadoras.
    A função descarta apenas cabeçalhos e textos técnicos.
    """

    texto = texto_limpo(valor)

    if not texto:
        return False

    texto_maiusculo = texto.upper()

    textos_tecnicos = [
        "REAJUSTE",
        "FATURA ATUAL",
        "FATURA REAJUSTADA",
        "PLANO",
        "FAIXA ETÁRIA",
        "FAIXA ETARIA",
        "VIDAS",
        "PER CAPITA",
        "TOTAL",
        "COTAÇÃO",
        "COTACAO",
        "DESCONTO",
        "DIFERENÇA",
        "DIFERENCA",
    ]

    for termo in textos_tecnicos:
        if termo in texto_maiusculo:
            return False

    return True


def localizar_elementos_proximos(
    ws,
    linha_inicial,
    coluna_inicial,
    altura=22,
    largura=12
):
    """
    Verifica quais indicadores estruturais existem
    próximo de um ponto identificado como OPERADORA.
    """

    elementos = set()

    linha_final = min(
        ws.max_row,
        linha_inicial + altura
    )

    coluna_final = min(
        ws.max_column,
        coluna_inicial + largura
    )

    indicadores = {
        "PLANO",
        "FAIXA ETÁRIA",
        "FAIXA ETARIA",
        "VIDAS",
        "PER CAPITA",
        "CUSTO MÉDIO",
        "CUSTO MEDIO",
        "TOTAL / FATURA",
    }

    for linha in range(
        linha_inicial,
        linha_final + 1
    ):
        for coluna in range(
            coluna_inicial,
            coluna_final + 1
        ):
            valor = texto_limpo(
                ws.cell(linha, coluna).value
            ).upper()

            if valor in indicadores:
                elementos.add(valor)

    return elementos


# ------------------------------------------------------------
# PREPARAR O ESTADO
# ------------------------------------------------------------

estado["operadoras"] = []
estado["operadora_atual"] = None
estado["cenarios_localizados"] = []
estado["operadoras_por_aba"] = OrderedDict()

# ------------------------------------------------------------
# ABRIR A MATRIZ
# ------------------------------------------------------------

wb = load_workbook(
    estado["arquivo_matriz"],
    data_only=True
)

operadoras_cotadas = OrderedDict()
operadoras_atuais_encontradas = OrderedDict()

# ------------------------------------------------------------
# ANALISAR SOMENTE AS ABAS DE COTAÇÃO JÁ IDENTIFICADAS
# ------------------------------------------------------------

for info_aba in estado["abas_cotacao"]:

    nome_aba = info_aba["aba"]
    ws = wb[nome_aba]

    # Segurança adicional:
    # nunca processar aba oculta ou veryHidden
    if ws.sheet_state != "visible":
        continue

    ocorrencias_da_aba = []

    # --------------------------------------------------------
    # LOCALIZAR TODAS AS CÉLULAS COM O TEXTO OPERADORA
    # --------------------------------------------------------

    for linha in ws.iter_rows():

        for celula in linha:

            valor_celula = texto_limpo(
                celula.value
            ).upper()

            if valor_celula != "OPERADORA":
                continue

            linha_rotulo = celula.row
            coluna_rotulo = celula.column

            # Na estrutura identificada, o nome fica
            # imediatamente à direita de OPERADORA
            valor_direita = ws.cell(
                linha_rotulo,
                coluna_rotulo + 1
            ).value

            nome_encontrado = texto_limpo(
                valor_direita
            )

            if not candidato_valido_operadora(
                nome_encontrado
            ):
                continue

            elementos_proximos = localizar_elementos_proximos(
                ws=ws,
                linha_inicial=linha_rotulo,
                coluna_inicial=coluna_rotulo
            )

            ocorrencia = {
                "aba": nome_aba,
                "operadora": nome_encontrado,
                "linha": linha_rotulo,
                "coluna_rotulo": coluna_rotulo,
                "coluna_nome": coluna_rotulo + 1,
                "elementos_proximos": sorted(
                    elementos_proximos
                ),
            }

            ocorrencias_da_aba.append(
                ocorrencia
            )

    # --------------------------------------------------------
    # CLASSIFICAR OPERADORA ATUAL E OPERADORAS COTADAS
    # --------------------------------------------------------
    #
    # Nesta matriz, o primeiro bloco válido da aba representa
    # o contrato atual. Os demais blocos representam cotações.
    #
    # A classificação é feita pela posição encontrada,
    # sem depender do nome da operadora.
    # --------------------------------------------------------

    if ocorrencias_da_aba:

        primeira_ocorrencia = ocorrencias_da_aba[0]
        nome_atual = primeira_ocorrencia["operadora"]

        primeira_ocorrencia["tipo"] = "operadora_atual"

        operadoras_atuais_encontradas[
            nome_atual
        ] = True

        estado["cenarios_localizados"].append(
            primeira_ocorrencia
        )

        for ocorrencia in ocorrencias_da_aba[1:]:

            ocorrencia["tipo"] = "operadora_cotada"

            estado["cenarios_localizados"].append(
                ocorrencia
            )

            nome_cotada = ocorrencia["operadora"]

            if nome_cotada not in operadoras_cotadas:
                operadoras_cotadas[
                    nome_cotada
                ] = True

    estado["operadoras_por_aba"][
        nome_aba
    ] = ocorrencias_da_aba

wb.close()

# ------------------------------------------------------------
# CONSOLIDAR RESULTADO
# ------------------------------------------------------------

estado["operadoras"] = list(
    operadoras_cotadas.keys()
)

if operadoras_atuais_encontradas:

    nomes_atuais = list(
        operadoras_atuais_encontradas.keys()
    )

    estado["operadora_atual"] = nomes_atuais[0]

    estado["operadoras_atuais_detectadas"] = (
        nomes_atuais
    )

else:

    estado["operadora_atual"] = None
    estado["operadoras_atuais_detectadas"] = []

# ------------------------------------------------------------
# EXIBIR RESULTADO
# ------------------------------------------------------------

print("\nOPERADORA ATUAL IDENTIFICADA:\n")

if estado["operadora_atual"]:
    print(
        f"🔵 {estado['operadora_atual']}"
    )
else:
    print("⚠️ Operadora atual não identificada.")

print("\n" + "-" * 70)
print("OPERADORAS DISPONÍVEIS PARA APRESENTAÇÃO")
print("-" * 70)

if estado["operadoras"]:

    for numero, operadora in enumerate(
        estado["operadoras"],
        start=1
    ):
        print(
            f"{numero}. ✅ {operadora}"
        )

else:

    print("⚠️ Nenhuma operadora cotada foi identificada.")

print("\n" + "-" * 70)
print("MAPEAMENTO POR ABA")
print("-" * 70)

for nome_aba, ocorrencias in (
    estado["operadoras_por_aba"].items()
):

    print(f"\n📄 {nome_aba}")

    for ocorrencia in ocorrencias:

        tipo = ocorrencia.get(
            "tipo",
            "não classificado"
        )

        print(
            f"  - {ocorrencia['operadora']}"
            f" | {tipo}"
            f" | linha {ocorrencia['linha']}"
            f" | coluna {ocorrencia['coluna_nome']}"
        )

print("\n" + "=" * 70)
print(
    "TOTAL DE OPERADORAS PARA APRESENTAÇÃO:",
    len(estado["operadoras"])
)
print("=" * 70)

MAPEANDO OPERADORAS E CENÁRIOS

OPERADORA ATUAL IDENTIFICADA:

🔵 Hapvida + Amil

----------------------------------------------------------------------
OPERADORAS DISPONÍVEIS PARA APRESENTAÇÃO
----------------------------------------------------------------------
1. ✅ Bradesco
2. ✅ Porto Linha P
3. ✅ Porto Linha Pro
4. ✅ SulAmérica I - De Para
5. ✅ Seguros Unimed
6. ✅ Porto Tradicional
7. ✅ Sulamerica II - Vital
8. ✅ Sulamerica III - 100

----------------------------------------------------------------------
MAPEAMENTO POR ABA
----------------------------------------------------------------------

📄 CC
  - Hapvida + Amil | operadora_atual | linha 3 | coluna 3
  - Bradesco | operadora_cotada | linha 22 | coluna 3
  - Porto Linha P | operadora_cotada | linha 22 | coluna 15
  - Porto Linha Pro | operadora_cotada | linha 44 | coluna 3
  - SulAmérica I - De Para | operadora_cotada | linha 44 | coluna 15
  - Seguros Unimed | operadora_cotada | linha 66 | coluna 3
  - Porto Tradicional | oper

In [5]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 5 - EXTRAIR DADOS COMPLETOS DOS CENÁRIOS
# ============================================================

from openpyxl import load_workbook
from collections import OrderedDict

print("=" * 70)
print("EXTRAINDO DADOS DOS CENÁRIOS")
print("=" * 70)


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def limpar_texto_5(valor):
    if valor is None:
        return ""

    return " ".join(
        str(valor).replace("\n", " ").split()
    ).strip()


def normalizar_texto_5(valor):
    texto = limpar_texto_5(valor).upper()

    texto = (
        texto
        .replace("Á", "A")
        .replace("À", "A")
        .replace("Â", "A")
        .replace("Ã", "A")
        .replace("É", "E")
        .replace("Ê", "E")
        .replace("Í", "I")
        .replace("Ó", "O")
        .replace("Ô", "O")
        .replace("Õ", "O")
        .replace("Ú", "U")
        .replace("Ç", "C")
    )

    return texto


def converter_numero_5(valor, padrao=0.0):
    if valor is None:
        return padrao

    if isinstance(valor, (int, float)):
        return float(valor)

    texto = limpar_texto_5(valor)

    if not texto:
        return padrao

    texto = (
        texto
        .replace("R$", "")
        .replace("%", "")
        .replace(" ", "")
    )

    if "." in texto and "," in texto:
        texto = texto.replace(".", "").replace(",", ".")

    elif "," in texto:
        texto = texto.replace(",", ".")

    try:
        return float(texto)

    except (TypeError, ValueError):
        return padrao


def converter_inteiro_5(valor, padrao=0):
    try:
        return int(
            round(
                converter_numero_5(
                    valor,
                    padrao
                )
            )
        )

    except (TypeError, ValueError):
        return padrao


def padronizar_faixa_5(valor):
    texto = normalizar_texto_5(valor)

    if not texto:
        return None

    texto = (
        texto
        .replace("ANOS", "")
        .replace("ANO", "")
        .replace("ACIMA DE", "")
        .replace("DE", "")
        .strip()
    )

    mapa_faixas = {
        "0 A 18": "00 - 18",
        "00 A 18": "00 - 18",
        "0 - 18": "00 - 18",
        "00 - 18": "00 - 18",

        "19 A 23": "19 - 23",
        "19 - 23": "19 - 23",

        "24 A 28": "24 - 28",
        "24 - 28": "24 - 28",

        "29 A 33": "29 - 33",
        "29 - 33": "29 - 33",

        "34 A 38": "34 - 38",
        "34 - 38": "34 - 38",

        "39 A 43": "39 - 43",
        "39 - 43": "39 - 43",

        "44 A 48": "44 - 48",
        "44 - 48": "44 - 48",

        "49 A 53": "49 - 53",
        "49 - 53": "49 - 53",

        "54 A 58": "54 - 58",
        "54 - 58": "54 - 58",

        "59+": "59+",
        "59 +": "59+",
        "59 OU MAIS": "59+",
        "59 MAIS": "59+",
        "59": "59+",
    }

    return mapa_faixas.get(texto)


def eh_rotulo_tecnico_5(valor):
    texto = normalizar_texto_5(valor)

    if not texto:
        return True

    rotulos = {
        "PLANO",
        "PRODUTO",
        "VIDAS",
        "PER CAPITA",
        "FAIXA ETARIA",
        "TOTAL",
        "CUSTO MEDIO",
        "TOTAL / FATURA",
        "TOTAL C DESCONTO",
        "DESCONTO",
        "OPERADORA",
    }

    return texto in rotulos


def localizar_linha_faixas_5(
    ws,
    linha_inicial,
    linha_final,
    coluna_inicial,
    coluna_final
):
    """
    Localiza a primeira linha do bloco que contém
    uma faixa etária reconhecida.
    """

    for linha in range(
        linha_inicial,
        linha_final + 1
    ):
        for coluna in range(
            coluna_inicial,
            coluna_final + 1
        ):
            faixa = padronizar_faixa_5(
                ws.cell(linha, coluna).value
            )

            if faixa is not None:
                return linha, coluna

    return None, None


def localizar_linha_cabecalho_5(
    ws,
    linha_faixas,
    coluna_inicial,
    coluna_final
):
    """
    Procura os cabeçalhos VIDAS e PER CAPITA
    imediatamente acima das faixas etárias.
    """

    linha_inicial = max(
        1,
        linha_faixas - 5
    )

    for linha in range(
        linha_faixas - 1,
        linha_inicial - 1,
        -1
    ):
        textos = [
            normalizar_texto_5(
                ws.cell(
                    linha,
                    coluna
                ).value
            )
            for coluna in range(
                coluna_inicial,
                coluna_final + 1
            )
        ]

        possui_vidas = any(
            texto == "VIDAS"
            for texto in textos
        )

        possui_per_capita = any(
            texto == "PER CAPITA"
            for texto in textos
        )

        if possui_vidas and possui_per_capita:
            return linha

    return None


def localizar_linha_planos_5(
    ws,
    linha_cabecalho,
    coluna_inicial,
    coluna_final
):
    """
    Procura a linha que contém os nomes dos planos.
    """

    linha_inicial = max(
        1,
        linha_cabecalho - 5
    )

    melhor_linha = None
    maior_quantidade = 0

    for linha in range(
        linha_cabecalho - 1,
        linha_inicial - 1,
        -1
    ):
        quantidade = 0

        for coluna in range(
            coluna_inicial,
            coluna_final + 1
        ):
            valor = limpar_texto_5(
                ws.cell(
                    linha,
                    coluna
                ).value
            )

            if (
                valor
                and not eh_rotulo_tecnico_5(valor)
                and not padronizar_faixa_5(valor)
            ):
                quantidade += 1

        if quantidade > maior_quantidade:
            maior_quantidade = quantidade
            melhor_linha = linha

    return melhor_linha


def localizar_colunas_produtos_5(
    ws,
    linha_cabecalho,
    coluna_inicial,
    coluna_final
):
    """
    Localiza pares de colunas VIDAS e PER CAPITA.
    """

    produtos = []

    coluna = coluna_inicial

    while coluna <= coluna_final:

        texto = normalizar_texto_5(
            ws.cell(
                linha_cabecalho,
                coluna
            ).value
        )

        if texto == "VIDAS":

            coluna_per_capita = None

            for deslocamento in range(1, 4):
                coluna_teste = coluna + deslocamento

                if coluna_teste > coluna_final:
                    break

                texto_teste = normalizar_texto_5(
                    ws.cell(
                        linha_cabecalho,
                        coluna_teste
                    ).value
                )

                if texto_teste == "PER CAPITA":
                    coluna_per_capita = coluna_teste
                    break

            if coluna_per_capita is not None:
                produtos.append({
                    "coluna_vidas": coluna,
                    "coluna_per_capita": coluna_per_capita,
                })

                coluna = coluna_per_capita

        coluna += 1

    return produtos


def localizar_total_fatura_5(
    ws,
    linha_inicial,
    linha_final,
    coluna_inicial,
    coluna_final
):
    """
    Procura TOTAL / FATURA dentro ou logo após o bloco.
    """

    limite_linha = min(
        ws.max_row,
        linha_final + 5
    )

    for linha in range(
        linha_inicial,
        limite_linha + 1
    ):
        for coluna in range(
            coluna_inicial,
            coluna_final + 1
        ):
            texto = normalizar_texto_5(
                ws.cell(
                    linha,
                    coluna
                ).value
            )

            if texto == "TOTAL / FATURA":

                candidatos = []

                for deslocamento_coluna in range(1, 5):
                    coluna_teste = (
                        coluna
                        + deslocamento_coluna
                    )

                    if coluna_teste > ws.max_column:
                        break

                    candidatos.append(
                        ws.cell(
                            linha,
                            coluna_teste
                        ).value
                    )

                for deslocamento_linha in range(1, 4):
                    linha_teste = (
                        linha
                        + deslocamento_linha
                    )

                    if linha_teste > ws.max_row:
                        break

                    candidatos.append(
                        ws.cell(
                            linha_teste,
                            coluna
                        ).value
                    )

                for candidato in candidatos:
                    numero = converter_numero_5(
                        candidato,
                        None
                    )

                    if numero is not None:
                        return numero

    return 0.0


def extrair_bloco_cenario_5(
    ws,
    ocorrencia,
    proxima_linha_bloco=None
):
    """
    Extrai um cenário completo usando o ponto OPERADORA
    encontrado na Etapa 4.
    """

    nome_operadora = ocorrencia["operadora"]
    linha_operadora = ocorrencia["linha"]
    coluna_nome = ocorrencia["coluna_nome"]

    # O bloco começa duas colunas antes do nome,
    # mas nunca abaixo da coluna 1.
    coluna_inicial = max(
        1,
        coluna_nome - 1
    )

    # A largura é determinada pelo próximo bloco horizontal
    # ou por uma janela máxima de 11 colunas.
    colunas_operadora_na_mesma_linha = []

    for outra in estado["operadoras_por_aba"][
        ws.title
    ]:
        if (
            outra["linha"] == linha_operadora
            and outra["coluna_nome"] > coluna_nome
        ):
            colunas_operadora_na_mesma_linha.append(
                outra["coluna_nome"]
            )

    if colunas_operadora_na_mesma_linha:
        proxima_coluna = min(
            colunas_operadora_na_mesma_linha
        )

        coluna_final = proxima_coluna - 2

    else:
        coluna_final = min(
            ws.max_column,
            coluna_nome + 10
        )

    if proxima_linha_bloco is not None:
        linha_final = proxima_linha_bloco - 1

    else:
        linha_final = min(
            ws.max_row,
            linha_operadora + 30
        )

    linha_faixas, coluna_faixas = (
        localizar_linha_faixas_5(
            ws=ws,
            linha_inicial=linha_operadora,
            linha_final=linha_final,
            coluna_inicial=coluna_inicial,
            coluna_final=coluna_final,
        )
    )

    resultado = {
        "operadora": nome_operadora,
        "aba": ws.title,
        "tipo": ocorrencia.get(
            "tipo",
            "operadora_cotada"
        ),
        "linha_inicio": linha_operadora,
        "linha_fim": linha_final,
        "coluna_inicio": coluna_inicial,
        "coluna_fim": coluna_final,
        "produtos": [],
        "faixas_etarias": [],
        "total_fatura": 0.0,
        "total_vidas": 0,
        "leitura_valida": False,
        "avisos": [],
    }

    if linha_faixas is None:
        resultado["avisos"].append(
            "Nenhuma faixa etária localizada."
        )

        return resultado

    linha_cabecalho = localizar_linha_cabecalho_5(
        ws=ws,
        linha_faixas=linha_faixas,
        coluna_inicial=coluna_inicial,
        coluna_final=coluna_final,
    )

    if linha_cabecalho is None:
        resultado["avisos"].append(
            "Cabeçalho VIDAS / PER CAPITA não localizado."
        )

        return resultado

    linha_planos = localizar_linha_planos_5(
        ws=ws,
        linha_cabecalho=linha_cabecalho,
        coluna_inicial=coluna_inicial,
        coluna_final=coluna_final,
    )

    colunas_produtos = localizar_colunas_produtos_5(
        ws=ws,
        linha_cabecalho=linha_cabecalho,
        coluna_inicial=coluna_inicial,
        coluna_final=coluna_final,
    )

    if not colunas_produtos:
        resultado["avisos"].append(
            "Nenhum par VIDAS / PER CAPITA localizado."
        )

        return resultado

    # --------------------------------------------------------
    # CRIAR PRODUTOS
    # --------------------------------------------------------

    for numero_produto, colunas in enumerate(
        colunas_produtos,
        start=1
    ):
        coluna_vidas = colunas["coluna_vidas"]
        coluna_per_capita = (
            colunas["coluna_per_capita"]
        )

        nome_plano = ""

        if linha_planos is not None:
            candidatos_nome = [
                ws.cell(
                    linha_planos,
                    coluna_vidas
                ).value,
                ws.cell(
                    linha_planos,
                    coluna_per_capita
                ).value,
            ]

            for candidato in candidatos_nome:
                candidato_limpo = limpar_texto_5(
                    candidato
                )

                if (
                    candidato_limpo
                    and not eh_rotulo_tecnico_5(
                        candidato_limpo
                    )
                ):
                    nome_plano = candidato_limpo
                    break

        if not nome_plano:
            nome_plano = (
                f"Produto {numero_produto}"
            )

        produto = {
            "plano": nome_plano,
            "coluna_vidas": coluna_vidas,
            "coluna_per_capita": coluna_per_capita,
            "faixas": [],
            "total_vidas": 0,
            "total": 0.0,
            "custo_medio": 0.0,
        }

        resultado["produtos"].append(
            produto
        )

    # --------------------------------------------------------
    # EXTRAIR FAIXAS
    # --------------------------------------------------------

    faixas_encontradas = OrderedDict()

    for linha in range(
        linha_faixas,
        linha_final + 1
    ):
        faixa = None

        for coluna in range(
            coluna_inicial,
            min(
                coluna_final,
                coluna_faixas + 2
            ) + 1
        ):
            faixa_teste = padronizar_faixa_5(
                ws.cell(
                    linha,
                    coluna
                ).value
            )

            if faixa_teste:
                faixa = faixa_teste
                break

        if faixa is None:
            continue

        if faixa not in faixas_encontradas:
            faixas_encontradas[faixa] = {
                "faixa": faixa,
                "linha": linha,
                "produtos": [],
            }

        for produto in resultado["produtos"]:

            vidas = converter_inteiro_5(
                ws.cell(
                    linha,
                    produto["coluna_vidas"]
                ).value
            )

            per_capita = converter_numero_5(
                ws.cell(
                    linha,
                    produto["coluna_per_capita"]
                ).value
            )

            total_faixa = vidas * per_capita

            produto["faixas"].append({
                "faixa": faixa,
                "vidas": vidas,
                "per_capita": per_capita,
                "total": total_faixa,
            })

            produto["total_vidas"] += vidas
            produto["total"] += total_faixa

            faixas_encontradas[
                faixa
            ]["produtos"].append({
                "plano": produto["plano"],
                "vidas": vidas,
                "per_capita": per_capita,
                "total": total_faixa,
            })

    # --------------------------------------------------------
    # CALCULAR CUSTOS MÉDIOS
    # --------------------------------------------------------

    for produto in resultado["produtos"]:

        if produto["total_vidas"] > 0:
            produto["custo_medio"] = (
                produto["total"]
                / produto["total_vidas"]
            )

        resultado["total_vidas"] += (
            produto["total_vidas"]
        )

    resultado["faixas_etarias"] = list(
        faixas_encontradas.values()
    )

    resultado["total_fatura"] = (
        localizar_total_fatura_5(
            ws=ws,
            linha_inicial=linha_operadora,
            linha_final=linha_final,
            coluna_inicial=coluna_inicial,
            coluna_final=coluna_final,
        )
    )

    soma_produtos = sum(
        produto["total"]
        for produto in resultado["produtos"]
    )

    resultado["soma_produtos"] = (
        soma_produtos
    )

    if resultado["total_fatura"] == 0:
        resultado["total_fatura"] = (
            soma_produtos
        )

        resultado["avisos"].append(
            "Total da fatura calculado pela soma dos produtos."
        )

    resultado["leitura_valida"] = (
        len(resultado["produtos"]) > 0
        and len(resultado["faixas_etarias"]) > 0
    )

    return resultado


# ============================================================
# PREPARAR ESTRUTURA CENTRAL
# ============================================================

estado["cenarios"] = {
    "com_coparticipacao": OrderedDict(),
    "sem_coparticipacao": OrderedDict(),
}

estado["diagnostico_cenarios"] = []


# ============================================================
# ABRIR MATRIZ
# ============================================================

wb = load_workbook(
    estado["arquivo_matriz"],
    data_only=True
)


# ============================================================
# PROCESSAR ABAS
# ============================================================

for nome_aba, ocorrencias in (
    estado["operadoras_por_aba"].items()
):

    ws = wb[nome_aba]

    if ws.sheet_state != "visible":
        continue

    nome_aba_normalizado = normalizar_texto_5(
        nome_aba
    )

    if (
        "S COPART" in nome_aba_normalizado
        or "SEM COPART" in nome_aba_normalizado
    ):
        modalidade = "sem_coparticipacao"

    elif (
        "C COPART" in nome_aba_normalizado
        or "COM COPART" in nome_aba_normalizado
    ):
        modalidade = "com_coparticipacao"

    else:
        modalidade = (
            f"aba_{len(estado['cenarios']) + 1}"
        )

        if modalidade not in estado["cenarios"]:
            estado["cenarios"][modalidade] = (
                OrderedDict()
            )

    # Usa somente as operadoras cotadas
    ocorrencias_cotadas = [
        item
        for item in ocorrencias
        if item.get("tipo") == "operadora_cotada"
    ]

    linhas_blocos = sorted(
        set(
            item["linha"]
            for item in ocorrencias_cotadas
        )
    )

    for ocorrencia in ocorrencias_cotadas:

        linha_atual = ocorrencia["linha"]

        proximas_linhas = [
            linha
            for linha in linhas_blocos
            if linha > linha_atual
        ]

        proxima_linha = (
            min(proximas_linhas)
            if proximas_linhas
            else None
        )

        dados_cenario = extrair_bloco_cenario_5(
            ws=ws,
            ocorrencia=ocorrencia,
            proxima_linha_bloco=proxima_linha,
        )

        nome_operadora = (
            dados_cenario["operadora"]
        )

        estado["cenarios"][
            modalidade
        ][nome_operadora] = dados_cenario

        estado["diagnostico_cenarios"].append({
            "modalidade": modalidade,
            "aba": nome_aba,
            "operadora": nome_operadora,
            "leitura_valida": dados_cenario[
                "leitura_valida"
            ],
            "produtos": len(
                dados_cenario["produtos"]
            ),
            "faixas": len(
                dados_cenario["faixas_etarias"]
            ),
            "total_vidas": dados_cenario[
                "total_vidas"
            ],
            "total_fatura": dados_cenario[
                "total_fatura"
            ],
            "avisos": dados_cenario[
                "avisos"
            ],
        })

wb.close()


# ============================================================
# EXIBIR RESULTADO
# ============================================================

for modalidade, cenarios in (
    estado["cenarios"].items()
):

    if not cenarios:
        continue

    print("\n" + "=" * 70)
    print(
        modalidade
        .replace("_", " ")
        .upper()
    )
    print("=" * 70)

    for operadora, dados in cenarios.items():

        status = (
            "✅"
            if dados["leitura_valida"]
            else "⚠️"
        )

        print(
            f"\n{status} {operadora}"
        )

        print(
            "  Produtos encontrados:",
            len(dados["produtos"])
        )

        print(
            "  Faixas encontradas:",
            len(dados["faixas_etarias"])
        )

        print(
            "  Total de vidas:",
            dados["total_vidas"]
        )

        print(
            "  Total da fatura:",
            f"R$ {dados['total_fatura']:,.2f}"
        )

        for produto in dados["produtos"]:

            print(
                f"    • {produto['plano']}"
                f" | vidas: {produto['total_vidas']}"
                f" | custo médio: "
                f"R$ {produto['custo_medio']:,.2f}"
                f" | total: "
                f"R$ {produto['total']:,.2f}"
            )

        if dados["avisos"]:

            print("  Avisos:")

            for aviso in dados["avisos"]:
                print(
                    f"    - {aviso}"
                )


print("\n" + "=" * 70)
print("RESUMO DA ETAPA 5")
print("=" * 70)

print(
    "Cenários com coparticipação:",
    len(
        estado["cenarios"][
            "com_coparticipacao"
        ]
    )
)

print(
    "Cenários sem coparticipação:",
    len(
        estado["cenarios"][
            "sem_coparticipacao"
        ]
    )
)

leituras_validas = sum(
    1
    for item in estado["diagnostico_cenarios"]
    if item["leitura_valida"]
)

print(
    "Leituras consideradas válidas:",
    leituras_validas
)

print("=" * 70)

EXTRAINDO DADOS DOS CENÁRIOS

ABA 3

✅ Bradesco
  Produtos encontrados: 5
  Faixas encontradas: 10
  Total de vidas: 24
  Total da fatura: R$ 13,191.99
    • Efetivo E | vidas: 6 | custo médio: R$ 533.31 | total: R$ 3,199.86
    • Efetivo E | vidas: 1 | custo médio: R$ 714.40 | total: R$ 714.40
    • Efetivo E | vidas: 4 | custo médio: R$ 736.98 | total: R$ 2,947.92
    • Efetivo A | vidas: 11 | custo médio: R$ 483.58 | total: R$ 5,319.36
    • Efetivo Plus E | vidas: 2 | custo médio: R$ 505.23 | total: R$ 1,010.45

✅ Porto Linha P
  Produtos encontrados: 5
  Faixas encontradas: 10
  Total de vidas: 24
  Total da fatura: R$ 7,916.67
    • P220 E | vidas: 6 | custo médio: R$ 323.94 | total: R$ 1,943.64
    • P220 E | vidas: 1 | custo médio: R$ 398.08 | total: R$ 398.08
    • P220 E | vidas: 4 | custo médio: R$ 436.94 | total: R$ 1,747.75
    • P220 A | vidas: 11 | custo médio: R$ 290.78 | total: R$ 3,198.60
    • P320 E | vidas: 2 | custo médio: R$ 314.30 | total: R$ 628.60

✅ Porto Lin

In [6]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 5.1 FINAL
# CONSOLIDAR MODALIDADES E CENÁRIOS VÁLIDOS
# ============================================================

import re
import unicodedata
from collections import OrderedDict

print("=" * 70)
print("CONSOLIDANDO CENÁRIOS VÁLIDOS")
print("=" * 70)


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def normalizar_51(valor):
    if valor is None:
        return ""

    texto = str(valor).strip()

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )

    texto = texto.upper()

    texto = re.sub(
        r"[^A-Z0-9]+",
        " ",
        texto
    )

    return " ".join(
        texto.split()
    )


def identificar_modalidade_51(valor):
    """
    Reconhece modalidade curta ou extensa:

    CC
    SC
    COM COPART
    SEM COPART
    C COPART
    S COPART
    """

    texto = normalizar_51(valor)

    if texto in {
        "SC",
        "SEM COPART",
        "SEM COPARTICIPACAO",
        "S COPART",
    }:
        return "sem_coparticipacao"

    if texto in {
        "CC",
        "COM COPART",
        "COM COPARTICIPACAO",
        "C COPART",
    }:
        return "com_coparticipacao"

    if "SEM COPART" in texto:
        return "sem_coparticipacao"

    if "COM COPART" in texto:
        return "com_coparticipacao"

    if "S COPART" in texto:
        return "sem_coparticipacao"

    if "C COPART" in texto:
        return "com_coparticipacao"

    return None


def localizar_nome_oficial_51(
    nome_lido,
    operadoras_oficiais
):
    """
    Compara o nome lido com as operadoras identificadas
    na Etapa 4.

    Permite corrigir apenas o sufixo visual "I".
    Exemplos:

    SulAmérica I -> SulAmérica

    Sufixos II, III ou posteriores não são aceitos
    quando não constam na lista oficial da Etapa 4.
    """

    nome_normalizado = normalizar_51(
        nome_lido
    )

    for nome_oficial in operadoras_oficiais:

        oficial_normalizado = normalizar_51(
            nome_oficial
        )

        if nome_normalizado == oficial_normalizado:
            return nome_oficial

        if nome_normalizado == (
            oficial_normalizado + " I"
        ):
            return nome_oficial

    return None


# ============================================================
# VALIDAÇÕES
# ============================================================

if "estado" not in globals():
    raise NameError(
        "A variável estado não existe. "
        "Execute novamente as etapas anteriores."
    )

if not estado.get("operadoras"):
    raise ValueError(
        "A lista de operadoras da Etapa 4 está vazia."
    )

if not estado.get("cenarios"):
    raise ValueError(
        "Os cenários da Etapa 5 estão vazios. "
        "Execute novamente a Etapa 5 antes desta célula."
    )


# ============================================================
# PRESERVAR RESULTADO BRUTO DA ETAPA 5
# ============================================================

cenarios_brutos = estado["cenarios"]

estado["cenarios_brutos_etapa_5"] = (
    cenarios_brutos
)

operadoras_oficiais = list(
    estado["operadoras"]
)


# ============================================================
# NOVA ESTRUTURA CONSOLIDADA
# ============================================================

cenarios_consolidados = {
    "com_coparticipacao": OrderedDict(),
    "sem_coparticipacao": OrderedDict(),
}

descartados = []
sem_modalidade = []


# ============================================================
# CONSOLIDAR TODOS OS GRUPOS EXTRAÍDOS
# ============================================================

for nome_grupo, grupo in cenarios_brutos.items():

    if not isinstance(grupo, dict):
        continue

    for chave_cenario, dados in grupo.items():

        if not isinstance(dados, dict):
            continue

        nome_lido = dados.get(
            "operadora",
            chave_cenario
        )

        nome_oficial = localizar_nome_oficial_51(
            nome_lido,
            operadoras_oficiais
        )

        if nome_oficial is None:

            descartados.append({
                "nome": nome_lido,
                "origem": dados.get(
                    "aba",
                    nome_grupo
                ),
                "motivo": (
                    "Não consta entre as operadoras "
                    "válidas da Etapa 4."
                ),
            })

            continue

        # A Etapa 5 armazenou CC ou SC no campo "aba".
        origem_modalidade = dados.get(
            "aba",
            nome_grupo
        )

        modalidade = identificar_modalidade_51(
            origem_modalidade
        )

        # Segunda tentativa usando a chave do grupo.
        if modalidade is None:
            modalidade = identificar_modalidade_51(
                nome_grupo
            )

        if modalidade is None:

            sem_modalidade.append({
                "operadora": nome_oficial,
                "origem": origem_modalidade,
            })

            continue

        # Evita sobrescrever silenciosamente um
        # cenário oficial já armazenado.
        if (
            nome_oficial
            in cenarios_consolidados[modalidade]
        ):

            descartados.append({
                "nome": nome_lido,
                "origem": origem_modalidade,
                "motivo": (
                    "Cenário duplicado para a mesma "
                    "operadora e modalidade."
                ),
            })

            continue

        dados["operadora_original"] = (
            nome_lido
        )

        dados["operadora"] = (
            nome_oficial
        )

        dados["modalidade"] = (
            modalidade
        )

        cenarios_consolidados[
            modalidade
        ][nome_oficial] = dados


# ============================================================
# REORDENAR PELA ORDEM DA ETAPA 4
# ============================================================

for modalidade in [
    "com_coparticipacao",
    "sem_coparticipacao",
]:

    ordenados = OrderedDict()

    for operadora in operadoras_oficiais:

        if (
            operadora
            in cenarios_consolidados[modalidade]
        ):
            ordenados[operadora] = (
                cenarios_consolidados[
                    modalidade
                ][operadora]
            )

    cenarios_consolidados[
        modalidade
    ] = ordenados


# ============================================================
# SALVAR RESULTADO NO ESTADO
# ============================================================

estado["cenarios"] = (
    cenarios_consolidados
)

estado["cenarios_descartados"] = (
    descartados
)

estado["cenarios_sem_modalidade"] = (
    sem_modalidade
)


# ============================================================
# OPERADORAS PRESENTES NAS DUAS MODALIDADES
# ============================================================

operadoras_cc = set(
    estado["cenarios"][
        "com_coparticipacao"
    ].keys()
)

operadoras_sc = set(
    estado["cenarios"][
        "sem_coparticipacao"
    ].keys()
)

estado["operadoras_disponiveis"] = [
    operadora
    for operadora in operadoras_oficiais
    if (
        operadora in operadoras_cc
        and operadora in operadoras_sc
    )
]


# ============================================================
# EXIBIR CENÁRIOS CONSOLIDADOS
# ============================================================

for modalidade in [
    "com_coparticipacao",
    "sem_coparticipacao",
]:

    if modalidade == "com_coparticipacao":
        titulo = "COM COPARTICIPAÇÃO"
    else:
        titulo = "SEM COPARTICIPAÇÃO"

    print("\n" + "=" * 70)
    print(titulo)
    print("=" * 70)

    cenarios = estado["cenarios"][
        modalidade
    ]

    if not cenarios:
        print(
            "⚠️ Nenhum cenário válido localizado."
        )

        continue

    for numero, operadora in enumerate(
        cenarios.keys(),
        start=1
    ):

        dados = cenarios[
            operadora
        ]

        produtos = dados.get(
            "produtos",
            []
        )

        total_vidas = dados.get(
            "total_vidas",
            0
        )

        total_fatura = dados.get(
            "total_fatura",
            0.0
        )

        print(
            f"{numero}. ✅ {operadora}"
            f" | produtos: {len(produtos)}"
            f" | vidas: {total_vidas}"
            f" | total: R$ {total_fatura:,.2f}"
        )


# ============================================================
# RESUMO FINAL
# ============================================================

print("\n" + "=" * 70)
print("RESUMO FINAL DA ETAPA 5")
print("=" * 70)

quantidade_cc = len(
    estado["cenarios"][
        "com_coparticipacao"
    ]
)

quantidade_sc = len(
    estado["cenarios"][
        "sem_coparticipacao"
    ]
)

quantidade_disponivel = len(
    estado["operadoras_disponiveis"]
)

print(
    "Cenários com coparticipação:",
    quantidade_cc
)

print(
    "Cenários sem coparticipação:",
    quantidade_sc
)

print(
    "Operadoras presentes nas duas modalidades:",
    quantidade_disponivel
)

print(
    "\nOPERADORAS DISPONÍVEIS PARA SELEÇÃO:"
)

for numero, operadora in enumerate(
    estado["operadoras_disponiveis"],
    start=1
):
    print(
        f"{numero}. ✅ {operadora}"
    )


# ============================================================
# EXIBIR DESCARTES
# ============================================================

if estado["cenarios_descartados"]:

    print("\nCENÁRIOS DESCARTADOS:")

    for item in estado[
        "cenarios_descartados"
    ]:

        print(
            f"❌ {item['nome']}"
            f" | origem: {item['origem']}"
            f" | {item['motivo']}"
        )


if estado["cenarios_sem_modalidade"]:

    print("\nCENÁRIOS SEM MODALIDADE:")

    for item in estado[
        "cenarios_sem_modalidade"
    ]:

        print(
            f"⚠️ {item['operadora']}"
            f" | origem: {item['origem']}"
        )


# ============================================================
# VALIDAÇÃO OBRIGATÓRIA
# ============================================================

esperado = len(
    operadoras_oficiais
)

if quantidade_cc != esperado:
    print(
        "\n⚠️ Atenção: a quantidade CC não "
        "corresponde à Etapa 4."
    )

if quantidade_sc != esperado:
    print(
        "⚠️ Atenção: a quantidade SC não "
        "corresponde à Etapa 4."
    )

if (
    quantidade_cc == esperado
    and quantidade_sc == esperado
    and quantidade_disponivel == esperado
):
    print(
        "\n✅ CONSOLIDAÇÃO CONCLUÍDA COM SUCESSO"
    )
else:
    print(
        "\n⚠️ CONSOLIDAÇÃO AINDA PRECISA DE REVISÃO"
    )

print("=" * 70)

CONSOLIDANDO CENÁRIOS VÁLIDOS

COM COPARTICIPAÇÃO
1. ✅ Bradesco | produtos: 5 | vidas: 24 | total: R$ 13,191.99
2. ✅ Porto Linha P | produtos: 5 | vidas: 24 | total: R$ 7,916.67
3. ✅ Porto Linha Pro | produtos: 5 | vidas: 24 | total: R$ 7,547.30
4. ✅ SulAmérica I - De Para | produtos: 5 | vidas: 24 | total: R$ 11,015.52
5. ✅ Seguros Unimed | produtos: 5 | vidas: 24 | total: R$ 13,277.35
6. ✅ Porto Tradicional | produtos: 5 | vidas: 24 | total: R$ 9,929.36
7. ✅ Sulamerica II - Vital | produtos: 1 | vidas: 24 | total: R$ 13,783.52
8. ✅ Sulamerica III - 100 | produtos: 1 | vidas: 24 | total: R$ 16,120.95

SEM COPARTICIPAÇÃO
1. ✅ Bradesco | produtos: 5 | vidas: 24 | total: R$ 18,071.26
2. ✅ Porto Linha P | produtos: 5 | vidas: 24 | total: R$ 10,960.41
3. ✅ Porto Linha Pro | produtos: 5 | vidas: 24 | total: R$ 10,781.83
4. ✅ SulAmérica I - De Para | produtos: 5 | vidas: 24 | total: R$ 15,645.34
5. ✅ Seguros Unimed | produtos: 5 | vidas: 24 | total: R$ 18,827.63
6. ✅ Porto Tradicional | prod

In [7]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 6 - SELEÇÃO DOS CENÁRIOS
# ============================================================

print("=" * 70)
print("SELEÇÃO DOS CENÁRIOS")
print("=" * 70)

cenarios_disponiveis = list(
    estado["operadoras_disponiveis"]
)

if not cenarios_disponiveis:
    raise ValueError(
        "Nenhum cenário disponível."
    )

print("\nCENÁRIOS ENCONTRADOS:\n")

for numero, nome in enumerate(
    cenarios_disponiveis,
    start=1
):
    print(f"{numero}. {nome}")

print(
    "\nSelecione até 4 cenários."
)

entrada = input(
    "\nDigite os números separados por vírgula: "
).strip()

numeros = []

for item in entrada.split(","):

    item = item.strip()

    if not item:
        continue

    numero = int(item)

    if (
        numero < 1
        or numero > len(cenarios_disponiveis)
    ):
        raise ValueError(
            f"Opção inválida: {numero}"
        )

    if numero not in numeros:
        numeros.append(numero)

if len(numeros) == 0:
    raise ValueError(
        "Nenhum cenário selecionado."
    )

if len(numeros) > 4:
    raise ValueError(
        "Selecione no máximo 4 cenários."
    )

cenarios_escolhidos = [
    cenarios_disponiveis[
        numero - 1
    ]
    for numero in numeros
]

estado["cenarios_escolhidos"] = (
    cenarios_escolhidos
)

print("\n" + "=" * 70)
print("CENÁRIOS SELECIONADOS")
print("=" * 70)

for ordem, nome in enumerate(
    cenarios_escolhidos,
    start=1
):
    print(
        f"{ordem}. ✅ {nome}"
    )

print("\nQuantidade:",
      len(cenarios_escolhidos))

print("\nETAPA 6 CONCLUÍDA")

SELEÇÃO DOS CENÁRIOS

CENÁRIOS ENCONTRADOS:

1. Bradesco
2. Porto Linha P
3. Porto Linha Pro
4. SulAmérica I - De Para
5. Seguros Unimed
6. Porto Tradicional
7. Sulamerica II - Vital
8. Sulamerica III - 100

Selecione até 4 cenários.

Digite os números separados por vírgula: 1,2,5,

CENÁRIOS SELECIONADOS
1. ✅ Bradesco
2. ✅ Porto Linha P
3. ✅ Seguros Unimed

Quantidade: 3

ETAPA 6 CONCLUÍDA


In [8]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 7 - LEITURA DINÂMICA DA BASE
# ELEGIBILIDADE + FAIXA ETÁRIA + PRODUTOS ATUAIS
# ============================================================

import re
import unicodedata
from collections import OrderedDict

from openpyxl import load_workbook

print("=" * 70)
print("LENDO A BASE DA MATRIZ")
print("=" * 70)


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def limpar_texto_7(valor):
    if valor is None:
        return ""

    return " ".join(
        str(valor).replace("\n", " ").split()
    ).strip()


def normalizar_7(valor):
    texto = limpar_texto_7(valor)

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )

    texto = texto.upper()

    texto = re.sub(
        r"[^A-Z0-9]+",
        " ",
        texto
    )

    return " ".join(
        texto.split()
    )


def converter_numero_7(valor, padrao=0.0):
    if valor is None:
        return padrao

    if isinstance(valor, (int, float)):
        return float(valor)

    texto = limpar_texto_7(valor)

    if not texto:
        return padrao

    texto = (
        texto
        .replace("R$", "")
        .replace("%", "")
        .replace(" ", "")
    )

    if "." in texto and "," in texto:
        texto = (
            texto
            .replace(".", "")
            .replace(",", ".")
        )

    elif "," in texto:
        texto = texto.replace(",", ".")

    try:
        return float(texto)

    except (TypeError, ValueError):
        return padrao


def padronizar_faixa_7(valor):
    texto = normalizar_7(valor)

    if not texto:
        return None

    mapa = {
        "0 18": "00 - 18",
        "00 18": "00 - 18",
        "0 A 18": "00 - 18",
        "00 A 18": "00 - 18",

        "19 23": "19 - 23",
        "19 A 23": "19 - 23",

        "24 28": "24 - 28",
        "24 A 28": "24 - 28",

        "29 33": "29 - 33",
        "29 A 33": "29 - 33",

        "34 38": "34 - 38",
        "34 A 38": "34 - 38",

        "39 43": "39 - 43",
        "39 A 43": "39 - 43",

        "44 48": "44 - 48",
        "44 A 48": "44 - 48",

        "49 53": "49 - 53",
        "49 A 53": "49 - 53",

        "54 58": "54 - 58",
        "54 A 58": "54 - 58",

        "59": "59+",
        "59 MAIS": "59+",
        "59 OU MAIS": "59+",
        "ACIMA DE 59": "59+",
        "ACIMA 59": "59+",
    }

    return mapa.get(texto)


def identificar_sexo_7(valor):
    texto = normalizar_7(valor)

    if texto in {
        "F",
        "FEMININO",
    }:
        return "Feminino"

    if texto in {
        "M",
        "MASCULINO",
    }:
        return "Masculino"

    return None


def identificar_vinculo_7(valor):
    texto = normalizar_7(valor)

    if texto in {
        "T",
        "TITULAR",
        "FUNCIONARIO",
        "FUNCIONARIA",
        "COLABORADOR",
        "COLABORADORA",
    }:
        return "Funcionário"

    if texto in {
        "D",
        "DEPENDENTE",
    }:
        return "Dependente"

    if "TITULAR" in texto:
        return "Funcionário"

    if "DEPEND" in texto:
        return "Dependente"

    return None


# ============================================================
# LOCALIZAR ABA BASE PELO CONTEÚDO
# ============================================================

def localizar_aba_base_7(wb):
    """
    Localiza uma aba visível com estrutura de cadastro de vidas.

    Não depende obrigatoriamente do nome 'Base'.
    """

    melhor_aba = None
    melhor_score = 0

    indicadores = {
        "NOME",
        "SEXO",
        "PLANO",
        "FAIXA ETARIA",
        "ELEGIBILIDADE",
        "IDADE",
        "VALOR",
    }

    for ws in wb.worksheets:

        if ws.sheet_state != "visible":
            continue

        score = 0

        limite_linhas = min(
            ws.max_row,
            30
        )

        limite_colunas = min(
            ws.max_column,
            50
        )

        encontrados = set()

        for linha in range(
            1,
            limite_linhas + 1
        ):
            for coluna in range(
                1,
                limite_colunas + 1
            ):
                texto = normalizar_7(
                    ws.cell(
                        linha,
                        coluna
                    ).value
                )

                if texto in indicadores:
                    encontrados.add(texto)

        score = len(encontrados)

        if score > melhor_score:
            melhor_score = score
            melhor_aba = ws

    if melhor_aba is None or melhor_score < 3:
        raise ValueError(
            "Nenhuma aba visível com estrutura de Base "
            "foi identificada."
        )

    return melhor_aba, melhor_score


# ============================================================
# LOCALIZAR LINHA DO CABEÇALHO
# ============================================================

def localizar_cabecalho_base_7(ws):
    """
    Procura a linha com maior quantidade de cabeçalhos úteis.
    """

    aliases = {
        "nome": {
            "NOME",
            "NOME BENEFICIARIO",
            "BENEFICIARIO",
            "VIDA",
        },

        "sexo": {
            "SEXO",
            "GENERO",
        },

        "vinculo": {
            "ELEGIBILIDADE",
            "VINCULO",
            "TIPO",
            "TITULAR DEPENDENTE",
        },

        "faixa": {
            "FAIXA ETARIA",
            "FAIXA",
        },

        "plano": {
            "PLANO",
            "PLANO 2",
            "PLANO2",
            "PRODUTO",
            "PRODUTO ATUAL",
            "PLANO ATUAL",
        },

        "valor": {
            "VALOR",
            "VALOR ATUAL",
            "MENSALIDADE",
            "PRECO",
        },

        "idade": {
            "IDADE",
        },
    }

    melhor_linha = None
    melhor_score = 0
    melhor_mapa = {}

    limite_linhas = min(
        ws.max_row,
        50
    )

    for linha in range(
        1,
        limite_linhas + 1
    ):
        mapa = {}

        for coluna in range(
            1,
            ws.max_column + 1
        ):
            texto = normalizar_7(
                ws.cell(
                    linha,
                    coluna
                ).value
            )

            if not texto:
                continue

            for campo, nomes in aliases.items():

                if texto in nomes:
                    # Quando existem cabeçalhos repetidos,
                    # preserva o último Plano e a primeira
                    # ocorrência dos demais campos.
                    if campo == "plano":
                        mapa[campo] = coluna

                    elif campo not in mapa:
                        mapa[campo] = coluna

        campos_essenciais = {
            "nome",
            "plano",
        }

        score = len(mapa)

        if (
            campos_essenciais.issubset(
                mapa.keys()
            )
            and score > melhor_score
        ):
            melhor_linha = linha
            melhor_score = score
            melhor_mapa = mapa

    if melhor_linha is None:
        raise ValueError(
            "O cabeçalho da Base não foi localizado."
        )

    return (
        melhor_linha,
        melhor_mapa,
        melhor_score
    )


# ============================================================
# DETERMINAR FIM DOS REGISTROS
# ============================================================

def linha_tem_dados_7(
    ws,
    linha,
    colunas
):
    for coluna in colunas.values():

        if coluna is None:
            continue

        valor = ws.cell(
            linha,
            coluna
        ).value

        if limpar_texto_7(valor):
            return True

    return False


# ============================================================
# VALIDAR ESTADO
# ============================================================

if "estado" not in globals():
    raise NameError(
        "O estado do projeto não existe. "
        "Execute novamente as etapas anteriores."
    )

arquivo_matriz = estado.get(
    "arquivo_matriz"
)

if not arquivo_matriz:
    raise ValueError(
        "O arquivo da matriz não está registrado."
    )


# ============================================================
# ABRIR MATRIZ E LOCALIZAR BASE
# ============================================================

wb = load_workbook(
    arquivo_matriz,
    data_only=True
)

ws_base, score_aba = localizar_aba_base_7(
    wb
)

linha_cabecalho, colunas, score_cabecalho = (
    localizar_cabecalho_base_7(
        ws_base
    )
)

print(
    "\nAba Base identificada:",
    ws_base.title
)

print(
    "Linha do cabeçalho:",
    linha_cabecalho
)

print("\nCOLUNAS IDENTIFICADAS:")

for campo, coluna in colunas.items():
    print(
        f"✅ {campo}: coluna {coluna}"
    )


# ============================================================
# LER REGISTROS
# ============================================================

registros = []

linhas_vazias_seguidas = 0

for linha in range(
    linha_cabecalho + 1,
    ws_base.max_row + 1
):

    if not linha_tem_dados_7(
        ws_base,
        linha,
        colunas
    ):
        linhas_vazias_seguidas += 1

        if linhas_vazias_seguidas >= 20:
            break

        continue

    linhas_vazias_seguidas = 0

    nome = limpar_texto_7(
        ws_base.cell(
            linha,
            colunas["nome"]
        ).value
    )

    plano = limpar_texto_7(
        ws_base.cell(
            linha,
            colunas["plano"]
        ).value
    )

    if not nome or not plano:
        continue

    sexo = None

    if colunas.get("sexo"):
        sexo = identificar_sexo_7(
            ws_base.cell(
                linha,
                colunas["sexo"]
            ).value
        )

    vinculo = None

    if colunas.get("vinculo"):
        vinculo = identificar_vinculo_7(
            ws_base.cell(
                linha,
                colunas["vinculo"]
            ).value
        )

    faixa = None

    if colunas.get("faixa"):
        faixa = padronizar_faixa_7(
            ws_base.cell(
                linha,
                colunas["faixa"]
            ).value
        )

    idade = None

    if colunas.get("idade"):
        idade_bruta = converter_numero_7(
            ws_base.cell(
                linha,
                colunas["idade"]
            ).value,
            None
        )

        if idade_bruta is not None:
            idade = int(
                round(idade_bruta)
            )

    # Caso a faixa não esteja pronta na Base,
    # calcula pela idade encontrada.
    if faixa is None and idade is not None:

        if idade <= 18:
            faixa = "00 - 18"

        elif idade <= 23:
            faixa = "19 - 23"

        elif idade <= 28:
            faixa = "24 - 28"

        elif idade <= 33:
            faixa = "29 - 33"

        elif idade <= 38:
            faixa = "34 - 38"

        elif idade <= 43:
            faixa = "39 - 43"

        elif idade <= 48:
            faixa = "44 - 48"

        elif idade <= 53:
            faixa = "49 - 53"

        elif idade <= 58:
            faixa = "54 - 58"

        else:
            faixa = "59+"

    valor = 0.0

    if colunas.get("valor"):
        valor = converter_numero_7(
            ws_base.cell(
                linha,
                colunas["valor"]
            ).value
        )

    registros.append({
        "linha": linha,
        "nome": nome,
        "sexo": sexo,
        "vinculo": vinculo,
        "idade": idade,
        "faixa_etaria": faixa,
        "plano": plano,
        "valor": valor,
    })

wb.close()

if not registros:
    raise ValueError(
        "Nenhum registro válido foi localizado na Base."
    )


# ============================================================
# CONSOLIDAR ELEGIBILIDADE
# ============================================================

total_vidas = len(
    registros
)

feminino = sum(
    1
    for registro in registros
    if registro["sexo"] == "Feminino"
)

masculino = sum(
    1
    for registro in registros
    if registro["sexo"] == "Masculino"
)

funcionarios = sum(
    1
    for registro in registros
    if registro["vinculo"] == "Funcionário"
)

dependentes = sum(
    1
    for registro in registros
    if registro["vinculo"] == "Dependente"
)

estado["registros_base"] = registros

estado["elegibilidade"] = {
    "total_vidas": total_vidas,
    "feminino": feminino,
    "masculino": masculino,
    "funcionarios": funcionarios,
    "dependentes": dependentes,
}


# ============================================================
# CONSOLIDAR FAIXA ETÁRIA
# ============================================================

ordem_faixas = [
    "00 - 18",
    "19 - 23",
    "24 - 28",
    "29 - 33",
    "34 - 38",
    "39 - 43",
    "44 - 48",
    "49 - 53",
    "54 - 58",
    "59+",
]

faixa_etaria = OrderedDict()

for faixa in ordem_faixas:
    faixa_etaria[faixa] = {
        "funcionarios": 0,
        "dependentes": 0,
        "sem_vinculo": 0,
        "total": 0,
    }

for registro in registros:

    faixa = registro["faixa_etaria"]

    if faixa not in faixa_etaria:
        continue

    faixa_etaria[faixa]["total"] += 1

    if registro["vinculo"] == "Funcionário":
        faixa_etaria[
            faixa
        ]["funcionarios"] += 1

    elif registro["vinculo"] == "Dependente":
        faixa_etaria[
            faixa
        ]["dependentes"] += 1

    else:
        faixa_etaria[
            faixa
        ]["sem_vinculo"] += 1

estado["faixa_etaria"] = faixa_etaria


# ============================================================
# CONSOLIDAR PRODUTOS ATUAIS
# ============================================================

produtos_atuais = OrderedDict()

for registro in registros:

    plano = registro["plano"]

    if plano not in produtos_atuais:

        produtos_atuais[plano] = {
            "plano": plano,
            "vidas": 0,
            "valor_total": 0.0,
            "percentual": 0.0,
        }

    produtos_atuais[
        plano
    ]["vidas"] += 1

    produtos_atuais[
        plano
    ]["valor_total"] += registro[
        "valor"
    ]

for plano, dados in produtos_atuais.items():

    if total_vidas > 0:
        dados["percentual"] = (
            dados["vidas"]
            / total_vidas
        )

estado["produtos_atuais"] = list(
    produtos_atuais.values()
)


# ============================================================
# EXIBIR ELEGIBILIDADE
# ============================================================

print("\n" + "=" * 70)
print("ELEGIBILIDADE")
print("=" * 70)

print(
    "População total:",
    total_vidas
)

print(
    "Feminino:",
    feminino
)

print(
    "Masculino:",
    masculino
)

print(
    "Funcionários:",
    funcionarios
)

print(
    "Dependentes:",
    dependentes
)


# ============================================================
# EXIBIR FAIXA ETÁRIA
# ============================================================

print("\n" + "=" * 70)
print("FAIXA ETÁRIA")
print("=" * 70)

for faixa, dados in faixa_etaria.items():

    print(
        f"{faixa}"
        f" | funcionários: "
        f"{dados['funcionarios']}"
        f" | dependentes: "
        f"{dados['dependentes']}"
        f" | total: "
        f"{dados['total']}"
    )


# ============================================================
# EXIBIR PRODUTOS ATUAIS
# ============================================================

print("\n" + "=" * 70)
print("PRODUTOS ATUAIS")
print("=" * 70)

for numero, produto in enumerate(
    estado["produtos_atuais"],
    start=1
):

    print(
        f"{numero}. {produto['plano']}"
        f" | vidas: {produto['vidas']}"
        f" | participação: "
        f"{produto['percentual']:.2%}"
        f" | valor total: R$ "
        f"{produto['valor_total']:,.2f}"
    )


# ============================================================
# VALIDAÇÕES DE CONSISTÊNCIA
# ============================================================

print("\n" + "=" * 70)
print("VALIDAÇÃO DA BASE")
print("=" * 70)

soma_sexo = (
    feminino
    + masculino
)

soma_vinculo = (
    funcionarios
    + dependentes
)

soma_faixas = sum(
    dados["total"]
    for dados in faixa_etaria.values()
)

soma_produtos = sum(
    produto["vidas"]
    for produto in estado[
        "produtos_atuais"
    ]
)

print(
    "Total da Base:",
    total_vidas
)

print(
    "Soma por sexo:",
    soma_sexo
)

print(
    "Soma por vínculo:",
    soma_vinculo
)

print(
    "Soma das faixas:",
    soma_faixas
)

print(
    "Soma dos produtos:",
    soma_produtos
)

if (
    soma_sexo == total_vidas
    and soma_vinculo == total_vidas
    and soma_faixas == total_vidas
    and soma_produtos == total_vidas
):

    print(
        "\n✅ ETAPA 7 CONCLUÍDA COM SUCESSO"
    )

else:

    print(
        "\n⚠️ ETAPA 7 CONCLUÍDA COM PENDÊNCIAS"
    )

    if soma_sexo != total_vidas:
        print(
            "- Existem registros sem sexo reconhecido."
        )

    if soma_vinculo != total_vidas:
        print(
            "- Existem registros sem vínculo reconhecido."
        )

    if soma_faixas != total_vidas:
        print(
            "- Existem registros sem faixa etária reconhecida."
        )

    if soma_produtos != total_vidas:
        print(
            "- A soma dos produtos não corresponde "
            "ao total da Base."
        )

print("=" * 70)

LENDO A BASE DA MATRIZ

Aba Base identificada: Base
Linha do cabeçalho: 1

COLUNAS IDENTIFICADAS:
✅ vinculo: coluna 1
✅ nome: coluna 2
✅ sexo: coluna 3
✅ idade: coluna 5
✅ faixa: coluna 8
✅ plano: coluna 9
✅ valor: coluna 10

ELEGIBILIDADE
População total: 24
Feminino: 13
Masculino: 11
Funcionários: 19
Dependentes: 5

FAIXA ETÁRIA
00 - 18 | funcionários: 0 | dependentes: 4 | total: 4
19 - 23 | funcionários: 1 | dependentes: 1 | total: 2
24 - 28 | funcionários: 2 | dependentes: 0 | total: 2
29 - 33 | funcionários: 3 | dependentes: 0 | total: 3
34 - 38 | funcionários: 2 | dependentes: 0 | total: 2
39 - 43 | funcionários: 4 | dependentes: 0 | total: 4
44 - 48 | funcionários: 3 | dependentes: 0 | total: 3
49 - 53 | funcionários: 2 | dependentes: 0 | total: 2
54 - 58 | funcionários: 0 | dependentes: 0 | total: 0
59+ | funcionários: 2 | dependentes: 0 | total: 2

PRODUTOS ATUAIS
1. ESSENCIAL 220E | vidas: 4 | participação: 16.67% | valor total: R$ 5,459.14
2. IDEAL 420A | vidas: 11 | partici

In [10]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 8A - UPLOAD E VALIDAÇÃO DO EXCEL ALIMENTADOR
# ============================================================

import os

from google.colab import files
from openpyxl import load_workbook

print("=" * 70)
print("UPLOAD DO EXCEL ALIMENTADOR")
print("=" * 70)

# ------------------------------------------------------------
# VALIDAR O ESTADO DO PROJETO
# ------------------------------------------------------------

if "estado" not in globals():
    raise NameError(
        "O estado do projeto não existe. "
        "Execute novamente as etapas anteriores."
    )

chaves_obrigatorias = [
    "cenarios_escolhidos",
    "cenarios",
    "elegibilidade",
    "faixa_etaria",
    "produtos_atuais",
]

chaves_faltantes = [
    chave
    for chave in chaves_obrigatorias
    if chave not in estado
]

if chaves_faltantes:
    raise ValueError(
        "As seguintes informações ainda não existem "
        "no estado do projeto: "
        + ", ".join(chaves_faltantes)
    )

if not estado["cenarios_escolhidos"]:
    raise ValueError(
        "Nenhum cenário foi selecionado na Etapa 6."
    )

if len(estado["cenarios_escolhidos"]) > 4:
    raise ValueError(
        "O modelo suporta no máximo quatro cenários."
    )

# ------------------------------------------------------------
# UPLOAD DO MODELO
# ------------------------------------------------------------

print(
    "\nSelecione o arquivo:\n"
    "Mezzo_Apresentação Saúde_2026_07_v1.xlsx\n"
)

uploaded_modelo = files.upload()

if not uploaded_modelo:
    raise ValueError(
        "Nenhum Excel Alimentador foi enviado."
    )

arquivos_xlsx = [
    nome
    for nome in uploaded_modelo.keys()
    if nome.lower().endswith(".xlsx")
]

if not arquivos_xlsx:
    raise ValueError(
        "Nenhum arquivo .xlsx foi identificado."
    )

arquivo_modelo = arquivos_xlsx[0]

estado["arquivo_modelo"] = arquivo_modelo

print("\nArquivo carregado:")
print(arquivo_modelo)

# ------------------------------------------------------------
# ABRIR E VALIDAR O MODELO
# ------------------------------------------------------------

wb_modelo = load_workbook(
    arquivo_modelo,
    data_only=False
)

abas_obrigatorias = [
    "Planos",
    "Produtos Atuais",
    "Elegibilidade",
    "Faixa Etária",
    "Equivalência",
    "Análise Financeira CC",
    "Impacto Financeiro CC",
    "Análise Financeira SC",
    "Impacto Financeiro SC",
]

abas_faltantes = [
    aba
    for aba in abas_obrigatorias
    if aba not in wb_modelo.sheetnames
]

print("\n" + "=" * 70)
print("ABAS DO MODELO")
print("=" * 70)

for aba in wb_modelo.sheetnames:

    if aba in abas_obrigatorias:
        print(f"✅ {aba}")

    else:
        print(f"ℹ️ {aba}")

if abas_faltantes:

    wb_modelo.close()

    raise ValueError(
        "O modelo não contém as seguintes abas "
        "obrigatórias: "
        + ", ".join(abas_faltantes)
    )

# ------------------------------------------------------------
# VALIDAR QUANTIDADE DE CENÁRIOS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CENÁRIOS QUE SERÃO INSERIDOS")
print("=" * 70)

for posicao, cenario in enumerate(
    estado["cenarios_escolhidos"],
    start=1
):
    print(
        f"{posicao}. ✅ {cenario}"
    )

# ------------------------------------------------------------
# VALIDAR OS CENÁRIOS CC E SC
# ------------------------------------------------------------

cenarios_cc = estado["cenarios"].get(
    "com_coparticipacao",
    {}
)

cenarios_sc = estado["cenarios"].get(
    "sem_coparticipacao",
    {}
)

cenarios_ausentes_cc = [
    cenario
    for cenario in estado["cenarios_escolhidos"]
    if cenario not in cenarios_cc
]

cenarios_ausentes_sc = [
    cenario
    for cenario in estado["cenarios_escolhidos"]
    if cenario not in cenarios_sc
]

if cenarios_ausentes_cc:

    wb_modelo.close()

    raise ValueError(
        "Os seguintes cenários não foram encontrados "
        "na modalidade CC: "
        + ", ".join(cenarios_ausentes_cc)
    )

if cenarios_ausentes_sc:

    wb_modelo.close()

    raise ValueError(
        "Os seguintes cenários não foram encontrados "
        "na modalidade SC: "
        + ", ".join(cenarios_ausentes_sc)
    )

# ------------------------------------------------------------
# REGISTRAR INFORMAÇÕES DO MODELO
# ------------------------------------------------------------

estado["modelo"] = {
    "arquivo": arquivo_modelo,
    "abas": list(
        wb_modelo.sheetnames
    ),
    "abas_obrigatorias": abas_obrigatorias,
    "limite_cenarios": 4,
}

wb_modelo.close()

# ------------------------------------------------------------
# RESULTADO
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDAÇÃO DO MODELO")
print("=" * 70)

print(
    "Arquivo:",
    estado["modelo"]["arquivo"]
)

print(
    "Abas obrigatórias encontradas:",
    len(abas_obrigatorias)
)

print(
    "Cenários selecionados:",
    len(
        estado["cenarios_escolhidos"]
    )
)

print(
    "Cenários CC disponíveis:",
    len(cenarios_cc)
)

print(
    "Cenários SC disponíveis:",
    len(cenarios_sc)
)

print(
    "\n✅ ETAPA 8A CONCLUÍDA COM SUCESSO"
)

print("=" * 70)

UPLOAD DO EXCEL ALIMENTADOR

Selecione o arquivo:
Mezzo_Apresentação Saúde_2026_07_v1.xlsx



Saving Mezzo_Apresentação Saúde_2026_07_v1.xlsx to Mezzo_Apresentação Saúde_2026_07_v1.xlsx

Arquivo carregado:
Mezzo_Apresentação Saúde_2026_07_v1.xlsx

ABAS DO MODELO
✅ Planos
✅ Produtos Atuais
✅ Elegibilidade
✅ Faixa Etária
✅ Equivalência
✅ Análise Financeira CC
✅ Impacto Financeiro CC
✅ Análise Financeira SC
✅ Impacto Financeiro SC
ℹ️ Economia

CENÁRIOS QUE SERÃO INSERIDOS
1. ✅ Bradesco
2. ✅ Porto Linha P
3. ✅ Seguros Unimed

VALIDAÇÃO DO MODELO
Arquivo: Mezzo_Apresentação Saúde_2026_07_v1.xlsx
Abas obrigatórias encontradas: 9
Cenários selecionados: 3
Cenários CC disponíveis: 8
Cenários SC disponíveis: 8

✅ ETAPA 8A CONCLUÍDA COM SUCESSO


In [11]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 8B
# PREENCHER PLANOS + PRODUTOS ATUAIS
# + ELEGIBILIDADE + FAIXA ETÁRIA
# ============================================================

import os
import re
import unicodedata
from copy import copy

from openpyxl import load_workbook
from openpyxl.cell.cell import MergedCell

print("=" * 70)
print("PREENCHENDO O EXCEL ALIMENTADOR")
print("=" * 70)


# ============================================================
# CONFIGURAÇÃO
# ============================================================

ARQUIVO_ENTRADA_8B = estado["arquivo_modelo"]

ARQUIVO_SAIDA_8B = (
    "Apresentador360_V2_Etapa8B.xlsx"
)

ABAS_NECESSARIAS_8B = [
    "Planos",
    "Produtos Atuais",
    "Elegibilidade",
    "Faixa Etária",
]


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def limpar_texto_8b(valor):
    if valor is None:
        return ""

    return " ".join(
        str(valor).replace("\n", " ").split()
    ).strip()


def normalizar_8b(valor):
    texto = limpar_texto_8b(valor)

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )

    texto = texto.upper()

    texto = re.sub(
        r"[^A-Z0-9]+",
        " ",
        texto
    )

    return " ".join(
        texto.split()
    )


def definir_valor_8b(
    ws,
    linha,
    coluna,
    valor
):
    """
    Escreve somente em células normais.

    Se a posição fizer parte de uma célula mesclada,
    localiza a célula principal do intervalo.
    """

    celula = ws.cell(
        linha,
        coluna
    )

    if not isinstance(
        celula,
        MergedCell
    ):
        celula.value = valor
        return

    for intervalo in ws.merged_cells.ranges:

        if celula.coordinate in intervalo:

            ws.cell(
                intervalo.min_row,
                intervalo.min_col
            ).value = valor

            return


def localizar_celulas_8b(
    ws,
    texto_procurado
):
    procurado = normalizar_8b(
        texto_procurado
    )

    encontrados = []

    for linha in ws.iter_rows():

        for celula in linha:

            if isinstance(
                celula,
                MergedCell
            ):
                continue

            texto = normalizar_8b(
                celula.value
            )

            if texto == procurado:

                encontrados.append({
                    "linha": celula.row,
                    "coluna": celula.column,
                    "coordenada": celula.coordinate,
                })

    return encontrados


def localizar_primeira_celula_8b(
    ws,
    textos
):
    for texto in textos:

        resultados = localizar_celulas_8b(
            ws,
            texto
        )

        if resultados:
            return resultados[0]

    return None


def copiar_estilo_8b(
    origem,
    destino
):
    """
    Copia apenas o estilo visual da célula.
    """

    if origem.has_style:
        destino._style = copy(
            origem._style
        )

    if origem.number_format:
        destino.number_format = (
            origem.number_format
        )

    if origem.alignment:
        destino.alignment = copy(
            origem.alignment
        )

    if origem.font:
        destino.font = copy(
            origem.font
        )

    if origem.fill:
        destino.fill = copy(
            origem.fill
        )

    if origem.border:
        destino.border = copy(
            origem.border
        )


def limpar_valores_area_8b(
    ws,
    linha_inicial,
    linha_final,
    coluna_inicial,
    coluna_final
):
    """
    Limpa valores, mas preserva toda a formatação.
    """

    for linha in range(
        linha_inicial,
        linha_final + 1
    ):
        for coluna in range(
            coluna_inicial,
            coluna_final + 1
        ):

            celula = ws.cell(
                linha,
                coluna
            )

            if isinstance(
                celula,
                MergedCell
            ):
                continue

            celula.value = None


def produtos_do_cenario_8b(
    modalidade,
    nome_cenario
):
    cenarios = estado["cenarios"].get(
        modalidade,
        {}
    )

    dados = cenarios.get(
        nome_cenario,
        {}
    )

    return dados.get(
        "produtos",
        []
    )


def nome_plano_8b(produto):
    nome = limpar_texto_8b(
        produto.get(
            "plano",
            ""
        )
    )

    return nome or "Produto"


def custo_medio_8b(produto):
    try:
        return float(
            produto.get(
                "custo_medio",
                0.0
            )
            or 0.0
        )

    except (TypeError, ValueError):
        return 0.0


# ============================================================
# VALIDAR ESTADO
# ============================================================

if "estado" not in globals():
    raise NameError(
        "O estado do projeto não existe."
    )

if not os.path.exists(
    ARQUIVO_ENTRADA_8B
):
    raise FileNotFoundError(
        "O Excel Alimentador não foi encontrado "
        "na sessão do Colab."
    )

cenarios_escolhidos = list(
    estado.get(
        "cenarios_escolhidos",
        []
    )
)

if not cenarios_escolhidos:
    raise ValueError(
        "Nenhum cenário foi escolhido."
    )

if len(cenarios_escolhidos) > 4:
    raise ValueError(
        "O modelo suporta até quatro cenários."
    )


# ============================================================
# ABRIR MODELO
# ============================================================

wb = load_workbook(
    ARQUIVO_ENTRADA_8B,
    data_only=False
)

for nome_aba in ABAS_NECESSARIAS_8B:

    if nome_aba not in wb.sheetnames:

        wb.close()

        raise ValueError(
            f"A aba '{nome_aba}' "
            "não existe no modelo."
        )


# ============================================================
# ABA PLANOS
# ============================================================

ws = wb["Planos"]

print("\nPreenchendo: Planos")


# ------------------------------------------------------------
# LOCALIZAR AS DUAS SEÇÕES DA ABA PLANOS
# ------------------------------------------------------------

marcadores_players = (
    localizar_celulas_8b(
        ws,
        "Players"
    )
)

marcadores_colagem = (
    localizar_celulas_8b(
        ws,
        "Players - Colagem"
    )
)

if not marcadores_players:
    raise ValueError(
        "O marcador 'Players' não foi localizado "
        "na aba Planos."
    )

if not marcadores_colagem:
    raise ValueError(
        "O marcador 'Players - Colagem' não foi "
        "localizado na aba Planos."
    )

inicio_players = marcadores_players[0]
inicio_colagem = marcadores_colagem[0]


# ------------------------------------------------------------
# FUNÇÃO PARA IDENTIFICAR BLOCOS VERTICAIS DE PLAYERS
# ------------------------------------------------------------

def localizar_blocos_planos_8b(
    ws,
    linha_inicial,
    linha_final,
    quantidade_maxima=5
):
    """
    Procura blocos que possuem:

    nome do player
    seguido de até dez linhas de produtos.

    Prioriza as posições já formatadas do modelo.
    """

    blocos = []

    for linha in range(
        linha_inicial,
        linha_final + 1
    ):

        valor = limpar_texto_8b(
            ws.cell(
                linha,
                1
            ).value
        )

        valor_b = limpar_texto_8b(
            ws.cell(
                linha,
                2
            ).value
        )

        candidato = (
            valor_b
            if valor_b
            else valor
        )

        if not candidato:
            continue

        texto = normalizar_8b(
            candidato
        )

        if texto in {
            "PLAYERS",
            "PLAYERS COLAGEM",
            "CUSTO MEDIO",
            "CUSTO MEDIO CC",
            "CUSTO MEDIO SC",
        }:
            continue

        produtos_abaixo = 0

        for deslocamento in range(
            1,
            11
        ):
            linha_teste = linha + deslocamento

            if linha_teste > linha_final:
                break

            possui_valor = False

            for coluna in range(
                1,
                min(
                    ws.max_column,
                    8
                ) + 1
            ):
                if limpar_texto_8b(
                    ws.cell(
                        linha_teste,
                        coluna
                    ).value
                ):
                    possui_valor = True
                    break

            if possui_valor:
                produtos_abaixo += 1

        if produtos_abaixo >= 5:

            blocos.append({
                "linha_nome": linha,
                "coluna_nome": (
                    2
                    if valor_b
                    else 1
                ),
            })

            if len(blocos) >= quantidade_maxima:
                break

    return blocos


linha_inicio_superior = (
    inicio_players["linha"] + 1
)

linha_fim_superior = (
    inicio_colagem["linha"] - 1
)

blocos_superiores = (
    localizar_blocos_planos_8b(
        ws=ws,
        linha_inicial=linha_inicio_superior,
        linha_final=linha_fim_superior,
        quantidade_maxima=5,
    )
)


# ------------------------------------------------------------
# FALLBACK POR ESTRUTURA DO MODELO
# ------------------------------------------------------------

if len(blocos_superiores) < 5:

    linhas_candidatas = []

    inicio = linha_inicio_superior

    # Cada player possui nome mais dez produtos.
    for indice in range(5):

        linha_nome = (
            inicio
            + indice * 11
        )

        if linha_nome <= linha_fim_superior:

            linhas_candidatas.append({
                "linha_nome": linha_nome,
                "coluna_nome": 2,
            })

    if len(linhas_candidatas) >= len(
        cenarios_escolhidos
    ):
        blocos_superiores = (
            linhas_candidatas
        )


# ------------------------------------------------------------
# OPERADORA ATUAL
# ------------------------------------------------------------

operadora_atual = limpar_texto_8b(
    estado.get(
        "operadora_atual",
        ""
    )
)

produtos_atuais = list(
    estado.get(
        "produtos_atuais",
        []
    )
)


# ------------------------------------------------------------
# PREPARAR PLAYERS
# ------------------------------------------------------------

players_planos = []

if operadora_atual:

    players_planos.append({
        "nome": operadora_atual,
        "produtos": [
            {
                "plano": produto.get(
                    "plano",
                    "Produto"
                ),
                "custo_medio": (
                    produto.get(
                        "valor_total",
                        0.0
                    )
                    / produto.get(
                        "vidas",
                        1
                    )
                    if produto.get(
                        "vidas",
                        0
                    ) > 0
                    else 0.0
                ),
            }
            for produto in produtos_atuais
        ],
        "atual": True,
    })

for cenario in cenarios_escolhidos:

    produtos_cc = produtos_do_cenario_8b(
        "com_coparticipacao",
        cenario
    )

    produtos_sc = produtos_do_cenario_8b(
        "sem_coparticipacao",
        cenario
    )

    produtos_combinados = []

    quantidade_produtos = max(
        len(produtos_cc),
        len(produtos_sc)
    )

    for indice in range(
        quantidade_produtos
    ):

        produto_cc = (
            produtos_cc[indice]
            if indice < len(produtos_cc)
            else {}
        )

        produto_sc = (
            produtos_sc[indice]
            if indice < len(produtos_sc)
            else {}
        )

        nome_plano = nome_plano_8b(
            produto_cc
            if produto_cc
            else produto_sc
        )

        produtos_combinados.append({
            "plano": nome_plano,
            "custo_medio_cc": (
                custo_medio_8b(
                    produto_cc
                )
            ),
            "custo_medio_sc": (
                custo_medio_8b(
                    produto_sc
                )
            ),
        })

    players_planos.append({
        "nome": cenario,
        "produtos": produtos_combinados,
        "atual": False,
    })


# ------------------------------------------------------------
# PREENCHER BLOCOS SUPERIORES
# ------------------------------------------------------------

for indice_bloco, bloco in enumerate(
    blocos_superiores[:5]
):

    linha_nome = bloco["linha_nome"]
    coluna_nome = bloco["coluna_nome"]

    if indice_bloco < len(
        players_planos
    ):

        player = players_planos[
            indice_bloco
        ]

        definir_valor_8b(
            ws,
            linha_nome,
            coluna_nome,
            player["nome"]
        )

        for indice_produto in range(
            10
        ):

            linha_produto = (
                linha_nome
                + 1
                + indice_produto
            )

            if indice_produto < len(
                player["produtos"]
            ):

                produto = player[
                    "produtos"
                ][indice_produto]

                definir_valor_8b(
                    ws,
                    linha_produto,
                    coluna_nome,
                    produto.get(
                        "plano",
                        "Produto"
                    )
                )

            else:

                definir_valor_8b(
                    ws,
                    linha_produto,
                    coluna_nome,
                    f"Produto {indice_produto + 1}"
                )

    else:

        definir_valor_8b(
            ws,
            linha_nome,
            coluna_nome,
            f"Player {indice_bloco + 1}"
        )


# ============================================================
# ABA PRODUTOS ATUAIS
# ============================================================

ws = wb["Produtos Atuais"]

print("Preenchendo: Produtos Atuais")

cabecalhos_plano = (
    localizar_celulas_8b(
        ws,
        "Plano Atual"
    )
)

if not cabecalhos_plano:
    raise ValueError(
        "Nenhuma tabela 'Plano Atual' foi encontrada."
    )

for tabela in cabecalhos_plano:

    linha_cabecalho = tabela["linha"]
    coluna_plano = tabela["coluna"]

    coluna_vidas = coluna_plano + 1
    coluna_terceira = coluna_plano + 2

    # Limpa até dez produtos.
    for deslocamento in range(
        1,
        11
    ):

        linha = (
            linha_cabecalho
            + deslocamento
        )

        definir_valor_8b(
            ws,
            linha,
            coluna_plano,
            None
        )

        definir_valor_8b(
            ws,
            linha,
            coluna_vidas,
            None
        )

        definir_valor_8b(
            ws,
            linha,
            coluna_terceira,
            None
        )

    for indice, produto in enumerate(
        produtos_atuais[:10],
        start=1
    ):

        linha = (
            linha_cabecalho
            + indice
        )

        plano = produto.get(
            "plano",
            ""
        )

        vidas = int(
            produto.get(
                "vidas",
                0
            )
        )

        percentual = float(
            produto.get(
                "percentual",
                0.0
            )
        )

        valor_total = float(
            produto.get(
                "valor_total",
                0.0
            )
        )

        definir_valor_8b(
            ws,
            linha,
            coluna_plano,
            plano
        )

        # Na tabela da direita, o modelo
        # costuma exibir "6 VIDAS".
        titulo_terceira = normalizar_8b(
            ws.cell(
                linha_cabecalho,
                coluna_terceira
            ).value
        )

        if titulo_terceira == "VALOR ATUAL":

            definir_valor_8b(
                ws,
                linha,
                coluna_vidas,
                f"{vidas} VIDAS"
            )

            definir_valor_8b(
                ws,
                linha,
                coluna_terceira,
                valor_total
            )

        else:

            definir_valor_8b(
                ws,
                linha,
                coluna_vidas,
                vidas
            )

            definir_valor_8b(
                ws,
                linha,
                coluna_terceira,
                percentual
            )

# Atualizar linhas de total localizadas.
celulas_total = localizar_celulas_8b(
    ws,
    "Total"
)

for item in celulas_total:

    linha = item["linha"]
    coluna = item["coluna"]

    if coluna + 1 <= ws.max_column:

        definir_valor_8b(
            ws,
            linha,
            coluna + 1,
            estado["elegibilidade"][
                "total_vidas"
            ]
        )

    if coluna + 2 <= ws.max_column:

        cabecalho_terceiro = ""

        for linha_teste in range(
            1,
            linha
        ):

            valor = normalizar_8b(
                ws.cell(
                    linha_teste,
                    coluna + 2
                ).value
            )

            if valor in {
                "VALOR ATUAL",
                "%",
            }:
                cabecalho_terceiro = valor

        if cabecalho_terceiro == "%":

            definir_valor_8b(
                ws,
                linha,
                coluna + 2,
                1.0
            )

        elif cabecalho_terceiro == "VALOR ATUAL":

            definir_valor_8b(
                ws,
                linha,
                coluna + 2,
                sum(
                    float(
                        produto.get(
                            "valor_total",
                            0.0
                        )
                    )
                    for produto in produtos_atuais
                )
            )


# ============================================================
# ABA ELEGIBILIDADE
# ============================================================

ws = wb["Elegibilidade"]

print("Preenchendo: Elegibilidade")

mapa_elegibilidade = {
    "POPULACAO TOTAL": estado[
        "elegibilidade"
    ]["total_vidas"],

    "FEMININO": estado[
        "elegibilidade"
    ]["feminino"],

    "MASCULINO": estado[
        "elegibilidade"
    ]["masculino"],

    "FUNCIONARIOS": estado[
        "elegibilidade"
    ]["funcionarios"],

    "DEPENDENTES": estado[
        "elegibilidade"
    ]["dependentes"],
}

campos_preenchidos = 0

for linha in range(
    1,
    ws.max_row + 1
):

    for coluna in range(
        1,
        ws.max_column + 1
    ):

        texto = normalizar_8b(
            ws.cell(
                linha,
                coluna
            ).value
        )

        if texto in mapa_elegibilidade:

            definir_valor_8b(
                ws,
                linha,
                coluna + 1,
                mapa_elegibilidade[
                    texto
                ]
            )

            campos_preenchidos += 1

if campos_preenchidos < 5:
    raise ValueError(
        "Nem todos os campos de Elegibilidade "
        "foram encontrados no modelo."
    )


# ============================================================
# ABA FAIXA ETÁRIA
# ============================================================

ws = wb["Faixa Etária"]

print("Preenchendo: Faixa Etária")

ordem_faixas = [
    "00 - 18",
    "19 - 23",
    "24 - 28",
    "29 - 33",
    "34 - 38",
    "39 - 43",
    "44 - 48",
    "49 - 53",
    "54 - 58",
    "59+",
]

faixas_preenchidas = 0

for linha in range(
    1,
    ws.max_row + 1
):

    valor_faixa = limpar_texto_8b(
        ws.cell(
            linha,
            1
        ).value
    )

    faixa_normalizada = normalizar_8b(
        valor_faixa
    )

    faixa_encontrada = None

    for faixa in ordem_faixas:

        if normalizar_8b(
            faixa
        ) == faixa_normalizada:

            faixa_encontrada = faixa
            break

    if faixa_encontrada is None:
        continue

    dados = estado["faixa_etaria"].get(
        faixa_encontrada,
        {}
    )

    definir_valor_8b(
        ws,
        linha,
        2,
        int(
            dados.get(
                "funcionarios",
                0
            )
        )
    )

    definir_valor_8b(
        ws,
        linha,
        3,
        int(
            dados.get(
                "dependentes",
                0
            )
        )
    )

    faixas_preenchidas += 1

if faixas_preenchidas != 10:
    raise ValueError(
        "Não foi possível preencher as dez "
        "faixas etárias."
    )


# ============================================================
# RECÁLCULO AO ABRIR NO EXCEL
# ============================================================

try:

    wb.calculation.fullCalcOnLoad = True
    wb.calculation.forceFullCalc = True
    wb.calculation.calcMode = "auto"

except Exception:

    try:

        wb.calculation_properties.fullCalcOnLoad = True
        wb.calculation_properties.forceFullCalc = True
        wb.calculation_properties.calcMode = "auto"

    except Exception:
        pass


# ============================================================
# SALVAR
# ============================================================

if os.path.exists(
    ARQUIVO_SAIDA_8B
):
    os.remove(
        ARQUIVO_SAIDA_8B
    )

wb.save(
    ARQUIVO_SAIDA_8B
)

wb.close()

estado["arquivo_etapa_8b"] = (
    ARQUIVO_SAIDA_8B
)


# ============================================================
# VALIDAR ARQUIVO GERADO
# ============================================================

wb_validacao = load_workbook(
    ARQUIVO_SAIDA_8B,
    data_only=False
)

print("\n" + "=" * 70)
print("RESUMO DA ETAPA 8B")
print("=" * 70)

print(
    "Arquivo criado:",
    ARQUIVO_SAIDA_8B
)

print(
    "Cenários escolhidos:",
    " | ".join(
        cenarios_escolhidos
    )
)

print(
    "Produtos atuais:",
    len(
        estado["produtos_atuais"]
    )
)

print(
    "Total de vidas:",
    estado["elegibilidade"][
        "total_vidas"
    ]
)

print(
    "Faixas preenchidas:",
    faixas_preenchidas
)

print(
    "Campos de elegibilidade preenchidos:",
    campos_preenchidos
)

print(
    "Abas preservadas:",
    len(
        wb_validacao.sheetnames
    )
)

wb_validacao.close()

print(
    "\n✅ ETAPA 8B CONCLUÍDA COM SUCESSO"
)

print("=" * 70)

PREENCHENDO O EXCEL ALIMENTADOR

Preenchendo: Planos
Preenchendo: Produtos Atuais
Preenchendo: Elegibilidade
Preenchendo: Faixa Etária

RESUMO DA ETAPA 8B
Arquivo criado: Apresentador360_V2_Etapa8B.xlsx
Cenários escolhidos: Bradesco | Porto Linha P | Seguros Unimed
Produtos atuais: 5
Total de vidas: 24
Faixas preenchidas: 10
Campos de elegibilidade preenchidos: 5
Abas preservadas: 10

✅ ETAPA 8B CONCLUÍDA COM SUCESSO


In [19]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 9
# MAPEAR ABAS E CAMPOS RESTANTES DO EXCEL ALIMENTADOR
# ============================================================

import os
import re
import unicodedata
from collections import Counter, defaultdict

from openpyxl import load_workbook
from openpyxl.cell.cell import MergedCell


print("=" * 70)
print("ETAPA 9 - MAPEANDO ABAS E CAMPOS RESTANTES")
print("=" * 70)


# ============================================================
# VALIDAR ESTADO DO PROJETO
# ============================================================

if "estado" not in globals():
    raise NameError(
        "O estado do projeto não existe. "
        "Execute as etapas anteriores antes da Etapa 9."
    )


ARQUIVO_ENTRADA_9 = estado.get(
    "arquivo_etapa_8b",
    "Apresentador360_V2_Etapa8B.xlsx"
)

if not os.path.exists(ARQUIVO_ENTRADA_9):
    raise FileNotFoundError(
        f"O arquivo '{ARQUIVO_ENTRADA_9}' não foi encontrado. "
        "Execute novamente a Etapa 8B."
    )


# ============================================================
# CONFIGURAÇÃO
# ============================================================

ABAS_JA_PREENCHIDAS_9 = {
    "Planos",
    "Produtos Atuais",
    "Elegibilidade",
    "Faixa Etária",
}

TERMOS_RELEVANTES_9 = [
    "coparticipação",
    "coparticipacao",
    "reembolso",
    "rede",
    "rede credenciada",
    "odontológico",
    "odontologico",
    "acomodação",
    "acomodacao",
    "abrangência",
    "abrangencia",
    "custeio",
    "contribuição",
    "contribuicao",
    "mensalidade",
    "valor",
    "custo",
    "impacto",
    "financeiro",
    "impacto financeiro",
    "impacto financeiro cc",
    "impacto financeiro sc",
    "plano",
    "produto",
    "operadora",
    "seguradora",
    "cenário",
    "cenario",
    "vidas",
    "titulares",
    "dependentes",
    "funcionários",
    "funcionarios",
    "elegibilidade",
    "compulsório",
    "compulsorio",
]


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def limpar_texto_9(valor):
    if valor is None:
        return ""

    return " ".join(
        str(valor).replace("\n", " ").split()
    ).strip()


def normalizar_9(valor):
    texto = limpar_texto_9(valor)

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )

    texto = texto.upper()

    texto = re.sub(
        r"[^A-Z0-9%]+",
        " ",
        texto
    )

    return " ".join(texto.split())


TERMOS_NORMALIZADOS_9 = {
    normalizar_9(termo)
    for termo in TERMOS_RELEVANTES_9
}


def celula_e_formula_9(celula):
    return (
        isinstance(celula.value, str)
        and celula.value.startswith("=")
    )


def celula_tem_texto_relevante_9(valor):
    texto = normalizar_9(valor)

    if not texto:
        return False

    for termo in TERMOS_NORMALIZADOS_9:
        if termo and termo in texto:
            return True

    return False


def obter_intervalo_mesclado_9(ws, coordenada):
    for intervalo in ws.merged_cells.ranges:
        if coordenada in intervalo:
            return str(intervalo)

    return None


def analisar_vizinhanca_9(
    ws,
    linha,
    coluna,
    raio_linhas=2,
    raio_colunas=3
):
    vizinhanca = []

    linha_inicial = max(
        1,
        linha - raio_linhas
    )

    linha_final = min(
        ws.max_row,
        linha + raio_linhas
    )

    coluna_inicial = max(
        1,
        coluna - raio_colunas
    )

    coluna_final = min(
        ws.max_column,
        coluna + raio_colunas
    )

    for linha_atual in range(
        linha_inicial,
        linha_final + 1
    ):
        for coluna_atual in range(
            coluna_inicial,
            coluna_final + 1
        ):
            celula = ws.cell(
                linha_atual,
                coluna_atual
            )

            if isinstance(
                celula,
                MergedCell
            ):
                continue

            valor = limpar_texto_9(
                celula.value
            )

            if valor:
                vizinhanca.append({
                    "coordenada": celula.coordinate,
                    "valor": valor,
                    "formula": celula_e_formula_9(
                        celula
                    ),
                })

    return vizinhanca


def localizar_celulas_vazias_proximas_9(
    ws,
    linha,
    coluna,
    distancia=3
):
    vazias = []

    # Prioriza células à direita do marcador.
    for deslocamento in range(
        1,
        distancia + 1
    ):
        coluna_teste = coluna + deslocamento

        if coluna_teste > ws.max_column:
            break

        celula = ws.cell(
            linha,
            coluna_teste
        )

        if isinstance(
            celula,
            MergedCell
        ):
            continue

        if celula.value is None:
            vazias.append(
                celula.coordinate
            )

    # Depois verifica células logo abaixo.
    for deslocamento in range(
        1,
        distancia + 1
    ):
        linha_teste = linha + deslocamento

        if linha_teste > ws.max_row:
            break

        celula = ws.cell(
            linha_teste,
            coluna
        )

        if isinstance(
            celula,
            MergedCell
        ):
            continue

        if celula.value is None:
            vazias.append(
                celula.coordinate
            )

    return vazias


def contar_celulas_preenchidas_9(ws):
    quantidade = 0

    for linha in ws.iter_rows():
        for celula in linha:
            if isinstance(
                celula,
                MergedCell
            ):
                continue

            if limpar_texto_9(
                celula.value
            ):
                quantidade += 1

    return quantidade


def contar_formulas_9(ws):
    quantidade = 0

    for linha in ws.iter_rows():
        for celula in linha:
            if isinstance(
                celula,
                MergedCell
            ):
                continue

            if celula_e_formula_9(
                celula
            ):
                quantidade += 1

    return quantidade


# ============================================================
# ABRIR O ARQUIVO GERADO NA ETAPA 8B
# ============================================================

wb = load_workbook(
    ARQUIVO_ENTRADA_9,
    data_only=False
)


# ============================================================
# IDENTIFICAR ABAS VISÍVEIS E OCULTAS
# ============================================================

abas_visiveis_9 = []

abas_ocultas_9 = []

for nome_aba in wb.sheetnames:
    ws = wb[nome_aba]

    if ws.sheet_state == "visible":
        abas_visiveis_9.append(
            nome_aba
        )
    else:
        abas_ocultas_9.append(
            nome_aba
        )


abas_restantes_9 = [
    nome_aba
    for nome_aba in abas_visiveis_9
    if nome_aba not in ABAS_JA_PREENCHIDAS_9
]


# ============================================================
# MAPEAR SOMENTE ABAS VISÍVEIS RESTANTES
# ============================================================

mapa_abas_9 = {}

total_marcadores_9 = 0

total_formulas_9 = 0

total_campos_candidatos_9 = 0


for nome_aba in abas_restantes_9:
    ws = wb[nome_aba]

    print(f"\nAnalisando aba: {nome_aba}")

    marcadores = []

    formulas = []

    campos_candidatos = []

    textos_repetidos = Counter()

    for linha in range(
        1,
        ws.max_row + 1
    ):
        for coluna in range(
            1,
            ws.max_column + 1
        ):
            celula = ws.cell(
                linha,
                coluna
            )

            if isinstance(
                celula,
                MergedCell
            ):
                continue

            valor = limpar_texto_9(
                celula.value
            )

            if not valor:
                continue

            texto_normalizado = normalizar_9(
                valor
            )

            textos_repetidos[
                texto_normalizado
            ] += 1

            if celula_e_formula_9(
                celula
            ):
                formulas.append({
                    "coordenada": celula.coordinate,
                    "formula": celula.value,
                })

            if celula_tem_texto_relevante_9(
                valor
            ):
                vazias_proximas = (
                    localizar_celulas_vazias_proximas_9(
                        ws=ws,
                        linha=linha,
                        coluna=coluna,
                        distancia=3,
                    )
                )

                intervalo_mesclado = (
                    obter_intervalo_mesclado_9(
                        ws,
                        celula.coordinate
                    )
                )

                marcador = {
                    "coordenada": celula.coordinate,
                    "texto": valor,
                    "texto_normalizado": texto_normalizado,
                    "intervalo_mesclado": intervalo_mesclado,
                    "celulas_vazias_proximas": vazias_proximas,
                    "vizinhanca": analisar_vizinhanca_9(
                        ws=ws,
                        linha=linha,
                        coluna=coluna,
                    ),
                }

                marcadores.append(
                    marcador
                )

                for coordenada in vazias_proximas:
                    campos_candidatos.append({
                        "marcador": valor,
                        "marcador_coordenada": (
                            celula.coordinate
                        ),
                        "campo_candidato": coordenada,
                    })

    mapa_abas_9[nome_aba] = {
        "estado_aba": ws.sheet_state,
        "max_linhas": ws.max_row,
        "max_colunas": ws.max_column,
        "celulas_preenchidas": (
            contar_celulas_preenchidas_9(
                ws
            )
        ),
        "quantidade_formulas": len(
            formulas
        ),
        "quantidade_marcadores": len(
            marcadores
        ),
        "quantidade_campos_candidatos": len(
            campos_candidatos
        ),
        "marcadores": marcadores,
        "formulas": formulas,
        "campos_candidatos": campos_candidatos,
        "textos_repetidos": dict(
            textos_repetidos
        ),
    }

    total_marcadores_9 += len(
        marcadores
    )

    total_formulas_9 += len(
        formulas
    )

    total_campos_candidatos_9 += len(
        campos_candidatos
    )

    print(
        "  Linhas:",
        ws.max_row
    )

    print(
        "  Colunas:",
        ws.max_column
    )

    print(
        "  Marcadores encontrados:",
        len(marcadores)
    )

    print(
        "  Fórmulas preservadas:",
        len(formulas)
    )

    print(
        "  Campos candidatos:",
        len(campos_candidatos)
    )


# ============================================================
# GUARDAR RESULTADO NO ESTADO
# ============================================================

estado["arquivo_entrada_etapa_9"] = (
    ARQUIVO_ENTRADA_9
)

estado["abas_visiveis"] = list(
    abas_visiveis_9
)

estado["abas_ocultas"] = list(
    abas_ocultas_9
)

estado["abas_restantes"] = list(
    abas_restantes_9
)

estado["mapa_etapa_9"] = (
    mapa_abas_9
)


# ============================================================
# CRIAR RELATÓRIO DE TEXTO
# ============================================================

ARQUIVO_RELATORIO_9 = (
    "Apresentador360_V2_Mapeamento_Etapa9.txt"
)

with open(
    ARQUIVO_RELATORIO_9,
    "w",
    encoding="utf-8"
) as arquivo:
    arquivo.write(
        "=" * 70 + "\n"
    )

    arquivo.write(
        "APRESENTADOR 360 V2 - MAPEAMENTO DA ETAPA 9\n"
    )

    arquivo.write(
        "=" * 70 + "\n\n"
    )

    arquivo.write(
        f"Arquivo analisado: {ARQUIVO_ENTRADA_9}\n"
    )

    arquivo.write(
        "Abas visíveis: "
        + " | ".join(abas_visiveis_9)
        + "\n"
    )

    arquivo.write(
        "Abas ocultas ignoradas: "
        + (
            " | ".join(abas_ocultas_9)
            if abas_ocultas_9
            else "Nenhuma"
        )
        + "\n"
    )

    arquivo.write(
        "Abas já preenchidas: "
        + " | ".join(
            sorted(
                ABAS_JA_PREENCHIDAS_9
            )
        )
        + "\n"
    )

    arquivo.write(
        "Abas restantes analisadas: "
        + (
            " | ".join(abas_restantes_9)
            if abas_restantes_9
            else "Nenhuma"
        )
        + "\n\n"
    )

    for nome_aba, dados_aba in mapa_abas_9.items():
        arquivo.write(
            "-" * 70 + "\n"
        )

        arquivo.write(
            f"ABA: {nome_aba}\n"
        )

        arquivo.write(
            "-" * 70 + "\n"
        )

        arquivo.write(
            f"Linhas: {dados_aba['max_linhas']}\n"
        )

        arquivo.write(
            f"Colunas: {dados_aba['max_colunas']}\n"
        )

        arquivo.write(
            "Células preenchidas: "
            f"{dados_aba['celulas_preenchidas']}\n"
        )

        arquivo.write(
            "Fórmulas: "
            f"{dados_aba['quantidade_formulas']}\n"
        )

        arquivo.write(
            "Marcadores: "
            f"{dados_aba['quantidade_marcadores']}\n"
        )

        arquivo.write(
            "Campos candidatos: "
            f"{dados_aba['quantidade_campos_candidatos']}\n\n"
        )

        arquivo.write(
            "MARCADORES ENCONTRADOS:\n"
        )

        if not dados_aba["marcadores"]:
            arquivo.write(
                "Nenhum marcador relevante encontrado.\n"
            )

        for marcador in dados_aba["marcadores"]:
            arquivo.write(
                f"- {marcador['coordenada']}: "
                f"{marcador['texto']}\n"
            )

            if marcador[
                "celulas_vazias_proximas"
            ]:
                arquivo.write(
                    "  Possíveis campos: "
                    + ", ".join(
                        marcador[
                            "celulas_vazias_proximas"
                        ]
                    )
                    + "\n"
                )

        arquivo.write("\n")


wb.close()


# ============================================================
# RESUMO
# ============================================================

print("\n" + "=" * 70)
print("RESUMO DA ETAPA 9")
print("=" * 70)

print(
    "Arquivo analisado:",
    ARQUIVO_ENTRADA_9
)

print(
    "Abas visíveis:",
    len(abas_visiveis_9)
)

print(
    "Abas ocultas ignoradas:",
    len(abas_ocultas_9)
)

print(
    "Abas já preenchidas:",
    len(
        [
            aba
            for aba in abas_visiveis_9
            if aba in ABAS_JA_PREENCHIDAS_9
        ]
    )
)

print(
    "Abas restantes analisadas:",
    len(abas_restantes_9)
)

print(
    "Marcadores encontrados:",
    total_marcadores_9
)

print(
    "Fórmulas preservadas:",
    total_formulas_9
)

print(
    "Campos candidatos encontrados:",
    total_campos_candidatos_9
)

print(
    "Relatório criado:",
    ARQUIVO_RELATORIO_9
)

if abas_restantes_9:
    print(
        "Abas restantes:",
        " | ".join(
            abas_restantes_9
        )
    )

if abas_ocultas_9:
    print(
        "Abas ocultas ignoradas:",
        " | ".join(
            abas_ocultas_9
        )
    )

print(
    "\n✅ ETAPA 9 CONCLUÍDA COM SUCESSO"
)

print("=" * 70)

ETAPA 9 - MAPEANDO ABAS E CAMPOS RESTANTES

Analisando aba: Equivalência
  Linhas: 51
  Colunas: 16
  Marcadores encontrados: 89
  Fórmulas preservadas: 55
  Campos candidatos: 240

Analisando aba: Análise Financeira CC
  Linhas: 51
  Colunas: 18
  Marcadores encontrados: 91
  Fórmulas preservadas: 245
  Campos candidatos: 132

Analisando aba: Impacto Financeiro CC
  Linhas: 8
  Colunas: 18
  Marcadores encontrados: 4
  Fórmulas preservadas: 39
  Campos candidatos: 3

Analisando aba: Análise Financeira SC
  Linhas: 51
  Colunas: 18
  Marcadores encontrados: 70
  Fórmulas preservadas: 245
  Campos candidatos: 114

Analisando aba: Impacto Financeiro SC
  Linhas: 8
  Colunas: 18
  Marcadores encontrados: 4
  Fórmulas preservadas: 39
  Campos candidatos: 3

Analisando aba: Economia
  Linhas: 11
  Colunas: 7
  Marcadores encontrados: 9
  Fórmulas preservadas: 13
  Campos candidatos: 15

RESUMO DA ETAPA 9
Arquivo analisado: Apresentador360_V2_Etapa8B.xlsx
Abas visíveis: 10
Abas ocultas ignor

In [26]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 10
# PREENCHER E VALIDAR A ABA EQUIVALÊNCIA
# ============================================================

import os
import re
import unicodedata

from openpyxl import load_workbook
from openpyxl.cell.cell import MergedCell


print("=" * 70)
print("ETAPA 10 - PREENCHENDO A ABA EQUIVALÊNCIA")
print("=" * 70)


# ============================================================
# VALIDAR ESTADO E ARQUIVO
# ============================================================

if "estado" not in globals():
    raise NameError(
        "O estado do projeto não existe. "
        "Execute as etapas anteriores."
    )


ARQUIVO_ENTRADA_10 = estado.get(
    "arquivo_etapa_8b",
    "Apresentador360_V2_Etapa8B.xlsx"
)

ARQUIVO_SAIDA_10 = "Apresentador360_V2_Etapa10.xlsx"

ARQUIVO_RELATORIO_10 = (
    "Apresentador360_V2_Relatorio_Etapa10.txt"
)


if not os.path.exists(ARQUIVO_ENTRADA_10):
    raise FileNotFoundError(
        f"O arquivo '{ARQUIVO_ENTRADA_10}' "
        "não foi encontrado."
    )


# ============================================================
# FUNÇÕES
# ============================================================

def limpar_texto_10(valor):
    if valor is None:
        return ""

    return " ".join(
        str(valor).replace("\n", " ").split()
    ).strip()


def normalizar_10(valor):
    texto = limpar_texto_10(valor)

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )

    texto = texto.upper()

    texto = re.sub(
        r"[^A-Z0-9]+",
        " ",
        texto
    )

    return " ".join(texto.split())


def eh_formula_10(valor):
    return (
        isinstance(valor, str)
        and valor.startswith("=")
    )


def eh_placeholder_10(valor):
    texto = normalizar_10(valor)

    return bool(
        re.fullmatch(
            r"PRODUTO [0-9]+",
            texto
        )
    )


def identificar_acomodacao_10(produto):
    """
    Identifica acomodação somente pelo sufixo
    explícito do produto.

    Exemplos:
    Compacto E   -> Enfermaria
    Efetivo A    -> Apartamento
    Essencial 220E -> Enfermaria
    """

    texto = normalizar_10(produto)

    if re.search(r"(?:\s|[0-9])E$", texto):
        return "Enfermaria"

    if re.search(r"(?:\s|[0-9])A$", texto):
        return "Apartamento"

    return None


def identificar_abrangencia_10(produto):
    """
    Preenche somente quando o nome do produto
    contém Nacional ou Regional.
    """

    texto = normalizar_10(produto)

    if "NACIONAL" in texto:
        return "Abrangência Nacional"

    if "REGIONAL" in texto:
        return "Abrangência Regional"

    return None


def escrever_se_vazio_10(
    ws,
    linha,
    coluna,
    valor
):
    """
    Escreve somente se:
    - houver valor identificado;
    - a célula estiver vazia;
    - a célula não possuir fórmula;
    - a célula não for parte secundária de mesclagem.
    """

    if valor is None:
        return "NÃO IDENTIFICADO"

    celula = ws.cell(
        linha,
        coluna
    )

    if isinstance(celula, MergedCell):
        return "MESCLAGEM PRESERVADA"

    if eh_formula_10(celula.value):
        return "FÓRMULA PRESERVADA"

    if limpar_texto_10(celula.value):
        return "VALOR EXISTENTE PRESERVADO"

    celula.value = valor

    return "PREENCHIDO"


# ============================================================
# ABRIR ARQUIVO
# ============================================================

wb = load_workbook(
    ARQUIVO_ENTRADA_10,
    data_only=False
)


for aba_obrigatoria in [
    "Planos",
    "Equivalência",
]:
    if aba_obrigatoria not in wb.sheetnames:
        wb.close()

        raise ValueError(
            f"A aba '{aba_obrigatoria}' "
            "não foi encontrada."
        )


ws_planos = wb["Planos"]
ws_equivalencia = wb["Equivalência"]


if ws_equivalencia.sheet_state != "visible":
    wb.close()

    raise ValueError(
        "A aba Equivalência está oculta."
    )


# ============================================================
# GUARDAR FÓRMULAS ANTES DO PREENCHIMENTO
# ============================================================

formulas_antes_10 = {}

for linha in ws_equivalencia.iter_rows():
    for celula in linha:

        if isinstance(celula, MergedCell):
            continue

        if eh_formula_10(celula.value):
            formulas_antes_10[
                celula.coordinate
            ] = celula.value


# ============================================================
# ESTRUTURA DO MODELO
# ============================================================
#
# Aba Planos:
# linhas 3 a 7 = players
# coluna B = nome do player
# colunas C a L = produtos 1 a 10
#
# Aba Equivalência:
# Produto 1 começa na linha 3
# Produto 2 começa na linha 8
# Produto 3 começa na linha 13
# ...
# Produto 10 começa na linha 48
#
# Linha inicial     = produto, já ligado por fórmula
# Linha inicial + 1 = acomodação
# Linha inicial + 2 = abrangência
# Linha inicial + 3 = reembolso
#
# Colunas B a F = players
# ============================================================

LINHAS_PLAYERS_PLANOS_10 = [
    3,
    4,
    5,
    6,
    7,
]

COLUNAS_PLAYERS_EQUIVALENCIA_10 = [
    2,
    3,
    4,
    5,
    6,
]

LINHAS_BLOCOS_EQUIVALENCIA_10 = [
    3,
    8,
    13,
    18,
    23,
    28,
    33,
    38,
    43,
    48,
]


# ============================================================
# PREENCHER EQUIVALÊNCIA
# ============================================================

registros_10 = []

total_produtos_validos_10 = 0
total_placeholders_10 = 0
total_acomodacoes_10 = 0
total_abrangencias_10 = 0
total_pendencias_reembolso_10 = 0


for indice_produto in range(10):

    coluna_produto_planos = (
        3 + indice_produto
    )

    linha_bloco = (
        LINHAS_BLOCOS_EQUIVALENCIA_10[
            indice_produto
        ]
    )

    linha_acomodacao = linha_bloco + 1
    linha_abrangencia = linha_bloco + 2
    linha_reembolso = linha_bloco + 3

    for indice_player in range(5):

        linha_player_planos = (
            LINHAS_PLAYERS_PLANOS_10[
                indice_player
            ]
        )

        coluna_player_equivalencia = (
            COLUNAS_PLAYERS_EQUIVALENCIA_10[
                indice_player
            ]
        )

        player = limpar_texto_10(
            ws_planos.cell(
                linha_player_planos,
                2
            ).value
        )

        produto = limpar_texto_10(
            ws_planos.cell(
                linha_player_planos,
                coluna_produto_planos
            ).value
        )

        if (
            not produto
            or eh_placeholder_10(produto)
        ):
            total_placeholders_10 += 1
            continue

        total_produtos_validos_10 += 1

        acomodacao = identificar_acomodacao_10(
            produto
        )

        abrangencia = identificar_abrangencia_10(
            produto
        )

        status_acomodacao = escrever_se_vazio_10(
            ws_equivalencia,
            linha_acomodacao,
            coluna_player_equivalencia,
            acomodacao
        )

        status_abrangencia = escrever_se_vazio_10(
            ws_equivalencia,
            linha_abrangencia,
            coluna_player_equivalencia,
            abrangencia
        )

        if status_acomodacao == "PREENCHIDO":
            total_acomodacoes_10 += 1

        if status_abrangencia == "PREENCHIDO":
            total_abrangencias_10 += 1

        celula_reembolso = ws_equivalencia.cell(
            linha_reembolso,
            coluna_player_equivalencia
        )

        if limpar_texto_10(
            celula_reembolso.value
        ):
            status_reembolso = (
                "VALOR EXISTENTE PRESERVADO"
            )
        else:
            status_reembolso = (
                "PENDENTE - SEM ORIGEM EXPLÍCITA"
            )

            total_pendencias_reembolso_10 += 1

        registros_10.append({
            "player": player,
            "produto_numero": indice_produto + 1,
            "produto": produto,
            "acomodacao": acomodacao,
            "status_acomodacao": status_acomodacao,
            "abrangencia": abrangencia,
            "status_abrangencia": status_abrangencia,
            "status_reembolso": status_reembolso,
            "celula_acomodacao": (
                ws_equivalencia.cell(
                    linha_acomodacao,
                    coluna_player_equivalencia
                ).coordinate
            ),
            "celula_abrangencia": (
                ws_equivalencia.cell(
                    linha_abrangencia,
                    coluna_player_equivalencia
                ).coordinate
            ),
            "celula_reembolso": (
                celula_reembolso.coordinate
            ),
        })


# ============================================================
# CONFIGURAR RECÁLCULO NO EXCEL
# ============================================================

try:
    wb.calculation.fullCalcOnLoad = True
    wb.calculation.forceFullCalc = True
    wb.calculation.calcMode = "auto"

except Exception:
    try:
        wb.calculation_properties.fullCalcOnLoad = True
        wb.calculation_properties.forceFullCalc = True
        wb.calculation_properties.calcMode = "auto"

    except Exception:
        pass


# ============================================================
# SALVAR NOVA VERSÃO
# ============================================================

if os.path.exists(ARQUIVO_SAIDA_10):
    os.remove(ARQUIVO_SAIDA_10)


wb.save(ARQUIVO_SAIDA_10)

wb.close()


# ============================================================
# VALIDAR FÓRMULAS DEPOIS DE SALVAR
# ============================================================

wb_validacao = load_workbook(
    ARQUIVO_SAIDA_10,
    data_only=False
)

ws_validacao = wb_validacao[
    "Equivalência"
]

formulas_depois_10 = {}

for linha in ws_validacao.iter_rows():
    for celula in linha:

        if isinstance(celula, MergedCell):
            continue

        if eh_formula_10(celula.value):
            formulas_depois_10[
                celula.coordinate
            ] = celula.value


formulas_preservadas_10 = (
    formulas_antes_10
    == formulas_depois_10
)

wb_validacao.close()


if not formulas_preservadas_10:
    raise RuntimeError(
        "As fórmulas da aba Equivalência "
        "foram alteradas."
    )


# ============================================================
# CRIAR RELATÓRIO
# ============================================================

with open(
    ARQUIVO_RELATORIO_10,
    "w",
    encoding="utf-8"
) as arquivo:

    arquivo.write(
        "=" * 70 + "\n"
    )

    arquivo.write(
        "APRESENTADOR 360 V2 - ETAPA 10\n"
    )

    arquivo.write(
        "PREENCHIMENTO DA ABA EQUIVALÊNCIA\n"
    )

    arquivo.write(
        "=" * 70 + "\n\n"
    )

    arquivo.write(
        "Arquivo de entrada: "
        + ARQUIVO_ENTRADA_10
        + "\n"
    )

    arquivo.write(
        "Arquivo de saída: "
        + ARQUIVO_SAIDA_10
        + "\n"
    )

    arquivo.write(
        "Produtos válidos: "
        + str(total_produtos_validos_10)
        + "\n"
    )

    arquivo.write(
        "Produtos vazios ou placeholders: "
        + str(total_placeholders_10)
        + "\n"
    )

    arquivo.write(
        "Acomodações preenchidas: "
        + str(total_acomodacoes_10)
        + "\n"
    )

    arquivo.write(
        "Abrangências preenchidas: "
        + str(total_abrangencias_10)
        + "\n"
    )

    arquivo.write(
        "Pendências de reembolso: "
        + str(total_pendencias_reembolso_10)
        + "\n"
    )

    arquivo.write(
        "Fórmulas preservadas: "
        + (
            "SIM"
            if formulas_preservadas_10
            else "NÃO"
        )
        + "\n\n"
    )

    arquivo.write(
        "-" * 70 + "\n"
    )

    arquivo.write(
        "DETALHAMENTO\n"
    )

    arquivo.write(
        "-" * 70 + "\n\n"
    )

    for registro in registros_10:

        arquivo.write(
            "Player: "
            + registro["player"]
            + "\n"
        )

        arquivo.write(
            "Produto "
            + str(registro["produto_numero"])
            + ": "
            + registro["produto"]
            + "\n"
        )

        arquivo.write(
            "Acomodação: "
            + (
                registro["acomodacao"]
                or "Não identificada"
            )
            + " | Célula: "
            + registro["celula_acomodacao"]
            + " | Status: "
            + registro["status_acomodacao"]
            + "\n"
        )

        arquivo.write(
            "Abrangência: "
            + (
                registro["abrangencia"]
                or "Não identificada"
            )
            + " | Célula: "
            + registro["celula_abrangencia"]
            + " | Status: "
            + registro["status_abrangencia"]
            + "\n"
        )

        arquivo.write(
            "Reembolso: Célula "
            + registro["celula_reembolso"]
            + " | Status: "
            + registro["status_reembolso"]
            + "\n\n"
        )


# ============================================================
# ATUALIZAR ESTADO
# ============================================================

estado["arquivo_etapa_10"] = (
    ARQUIVO_SAIDA_10
)

estado["relatorio_etapa_10"] = (
    ARQUIVO_RELATORIO_10
)

estado["registros_equivalencia"] = (
    registros_10
)

estado["formulas_etapa_10_preservadas"] = (
    formulas_preservadas_10
)


# ============================================================
# RESUMO
# ============================================================

print("\n" + "=" * 70)
print("RESUMO DA ETAPA 10")
print("=" * 70)

print(
    "Arquivo criado:",
    ARQUIVO_SAIDA_10
)

print(
    "Relatório criado:",
    ARQUIVO_RELATORIO_10
)

print(
    "Produtos válidos:",
    total_produtos_validos_10
)

print(
    "Produtos vazios ou placeholders:",
    total_placeholders_10
)

print(
    "Acomodações preenchidas:",
    total_acomodacoes_10
)

print(
    "Abrangências preenchidas:",
    total_abrangencias_10
)

print(
    "Pendências de reembolso:",
    total_pendencias_reembolso_10
)

print(
    "Fórmulas preservadas:",
    "SIM" if formulas_preservadas_10 else "NÃO"
)

print(
    "\n✅ ETAPA 10 CONCLUÍDA COM SUCESSO"
)

print("=" * 70)

ETAPA 10 - PREENCHENDO A ABA EQUIVALÊNCIA

RESUMO DA ETAPA 10
Arquivo criado: Apresentador360_V2_Etapa10.xlsx
Relatório criado: Apresentador360_V2_Relatorio_Etapa10.txt
Produtos válidos: 20
Produtos vazios ou placeholders: 30
Acomodações preenchidas: 20
Abrangências preenchidas: 4
Pendências de reembolso: 0
Fórmulas preservadas: SIM

✅ ETAPA 10 CONCLUÍDA COM SUCESSO


In [28]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 11
# VALIDAÇÃO DAS ABAS FINANCEIRAS
# ============================================================

import os

from openpyxl import load_workbook


print("=" * 70)
print("ETAPA 11 - VALIDAÇÃO FINANCEIRA")
print("=" * 70)


ARQUIVO_ENTRADA_11 = (
    estado.get(
        "arquivo_etapa_10",
        "Apresentador360_V2_Etapa10.xlsx"
    )
)

ARQUIVO_RELATORIO_11 = (
    "Apresentador360_V2_Relatorio_Etapa11.txt"
)


if not os.path.exists(
    ARQUIVO_ENTRADA_11
):
    raise FileNotFoundError(
        ARQUIVO_ENTRADA_11
    )


wb = load_workbook(
    ARQUIVO_ENTRADA_11,
    data_only=False
)


ABAS_ANALISADAS = [
    "Análise Financeira CC",
    "Impacto Financeiro CC",
    "Análise Financeira SC",
    "Impacto Financeiro SC",
    "Economia",
]


erros = []


for nome_aba in ABAS_ANALISADAS:

    ws = wb[nome_aba]

    for linha in ws.iter_rows():

        for celula in linha:

            valor = celula.value

            if not isinstance(
                valor,
                str
            ):
                continue

            valor_upper = valor.upper()

            if (
                "#REF!" in valor_upper
                or "#NAME?" in valor_upper
                or "#VALUE!" in valor_upper
                or "#DIV/0!" in valor_upper
                or "#N/A" in valor_upper
            ):

                erros.append({
                    "aba": nome_aba,
                    "celula": celula.coordinate,
                    "valor": valor,
                })


with open(
    ARQUIVO_RELATORIO_11,
    "w",
    encoding="utf-8"
) as arquivo:

    arquivo.write(
        "VALIDAÇÃO FINANCEIRA\n\n"
    )

    if not erros:

        arquivo.write(
            "NENHUM ERRO IDENTIFICADO\n"
        )

    else:

        for erro in erros:

            arquivo.write(
                f"Aba: {erro['aba']}\n"
            )

            arquivo.write(
                f"Célula: {erro['celula']}\n"
            )

            arquivo.write(
                f"Valor: {erro['valor']}\n\n"
            )


wb.close()


estado["relatorio_etapa_11"] = (
    ARQUIVO_RELATORIO_11
)

estado["erros_etapa_11"] = erros


print("\n" + "=" * 70)
print("RESUMO")
print("=" * 70)

print(
    "Abas verificadas:",
    len(ABAS_ANALISADAS)
)

print(
    "Erros encontrados:",
    len(erros)
)

print(
    "Relatório:",
    ARQUIVO_RELATORIO_11
)

print(
    "\n✅ ETAPA 11 CONCLUÍDA"
)

print("=" * 70)

ETAPA 11 - VALIDAÇÃO FINANCEIRA

RESUMO
Abas verificadas: 5
Erros encontrados: 0
Relatório: Apresentador360_V2_Relatorio_Etapa11.txt

✅ ETAPA 11 CONCLUÍDA


In [30]:
!pip install python-pptx -q

from pptx import Presentation

print("python-pptx OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 10.8 MB/s eta 0:00:00
python-pptx OK


In [34]:
from google.colab import files

uploaded = files.upload()

Saving Mezzo_Apresentação Saúde_2026_07_v1.pptx to Mezzo_Apresentação Saúde_2026_07_v1.pptx


In [36]:
import os

for arquivo in os.listdir():
    if arquivo.lower().endswith(".pptx"):
        print(arquivo)

Mezzo_Apresentação Saúde_2026_07_v1.pptx


In [37]:
from pptx import Presentation

ARQUIVO_PPT = "Mezzo_Apresentação Saúde_2026_07_v1.pptx"

ppt = Presentation(ARQUIVO_PPT)

for i, slide in enumerate(ppt.slides, start=1):

    print("\n" + "=" * 60)
    print(f"SLIDE {i}")

    contador = 0

    for shape in slide.shapes:

        if hasattr(shape, "text"):

            texto = shape.text.strip()

            if texto:

                contador += 1

                print(f"[{contador}] {texto[:300]}")


SLIDE 1
[1] Contar com a
[2] aggrega
[3] faz mais sentido!
[4] Mezzo

SLIDE 2
[1] Desenhos de projetos eficientes e disruptivos
[2] Indicadores: Medicina por capacidade de antecipar ocorrências e alto impacto no sinistro
[3] Operadores: Estrutura médica de referência para melhor gestão do sinistro
[4] Telemedicina Aggrega: Conveniência para o usuário e dados para melhor tomada de decisão no direcionamento assistencial
[5] Portal do colaborador desonerando atividades operacionais do RH
[6] Rede Aggrega: Clube de parceiros de custo mais equilibrado e desfechos eficientes
[7] ISC: Índice de Saúde Corporativa evolução do BIG DATA
[8] GSA: Programas de qualidade de vida centrado no usuário: bem estar e aculturamento corporativo

SLIDE 3
[1] NOSSOS SERVIÇOS
[2] Na Aggrega, pessoas, processos e tecnologias são vocacionadas dentro de nossa metodologia que busca produzir melhor, com redução de recursos e tempo, incorporando mais valor ao cliente. 

Isso é possível através de técnicas inovadora

In [38]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 12
# MAPEAR OBJETOS DO POWERPOINT
# ============================================================

import os

from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE


print("=" * 70)
print("ETAPA 12 - MAPEANDO OBJETOS DO POWERPOINT")
print("=" * 70)


# ============================================================
# CONFIGURAÇÃO
# ============================================================

ARQUIVO_PPT_ENTRADA_12 = (
    "Mezzo_Apresentação Saúde_2026_07_v1.pptx"
)

ARQUIVO_RELATORIO_12 = (
    "Apresentador360_V2_Mapeamento_PPT_Etapa12.txt"
)


if not os.path.exists(
    ARQUIVO_PPT_ENTRADA_12
):
    raise FileNotFoundError(
        "O PowerPoint não foi encontrado: "
        + ARQUIVO_PPT_ENTRADA_12
    )


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def limpar_texto_12(valor):

    if valor is None:
        return ""

    return " ".join(
        str(valor).replace("\n", " ").split()
    ).strip()


def emu_para_cm_12(valor):

    try:
        return round(
            valor / 360000,
            2
        )

    except Exception:
        return 0.0


def nome_tipo_shape_12(shape):

    mapa_tipos = {
        MSO_SHAPE_TYPE.AUTO_SHAPE: "FORMA",
        MSO_SHAPE_TYPE.CHART: "GRÁFICO",
        MSO_SHAPE_TYPE.GROUP: "GRUPO",
        MSO_SHAPE_TYPE.LINE: "LINHA",
        MSO_SHAPE_TYPE.PICTURE: "IMAGEM",
        MSO_SHAPE_TYPE.PLACEHOLDER: "PLACEHOLDER",
        MSO_SHAPE_TYPE.TABLE: "TABELA",
        MSO_SHAPE_TYPE.TEXT_BOX: "CAIXA DE TEXTO",
        MSO_SHAPE_TYPE.MEDIA: "MÍDIA",
    }

    return mapa_tipos.get(
        shape.shape_type,
        "OUTRO"
    )


def extrair_texto_shape_12(shape):

    if not hasattr(
        shape,
        "text"
    ):
        return ""

    return limpar_texto_12(
        shape.text
    )


def extrair_tabela_12(shape):

    if not getattr(
        shape,
        "has_table",
        False
    ):
        return []

    linhas = []

    tabela = shape.table

    for linha in tabela.rows:

        valores = []

        for celula in linha.cells:

            valores.append(
                limpar_texto_12(
                    celula.text
                )
            )

        linhas.append(
            valores
        )

    return linhas


def extrair_grafico_12(shape):

    if not getattr(
        shape,
        "has_chart",
        False
    ):
        return None

    grafico = shape.chart

    informacoes = {
        "tipo": str(
            grafico.chart_type
        ),
        "titulo": "",
        "series": [],
    }

    try:

        if grafico.has_title:

            informacoes["titulo"] = (
                limpar_texto_12(
                    grafico.chart_title.text_frame.text
                )
            )

    except Exception:
        pass

    try:

        for serie in grafico.series:

            informacoes["series"].append(
                limpar_texto_12(
                    serie.name
                )
            )

    except Exception:
        pass

    return informacoes


def extrair_vinculos_12(shape):

    vinculos = []

    try:

        for relacionamento in (
            shape.part.rels.values()
        ):

            alvo = str(
                relacionamento.target_ref
            )

            alvo_lower = alvo.lower()

            if (
                ".xlsx" in alvo_lower
                or ".xls" in alvo_lower
                or "oleobject" in alvo_lower
                or "external" in alvo_lower
            ):

                vinculos.append(
                    alvo
                )

    except Exception:
        pass

    return sorted(
        set(vinculos)
    )


# ============================================================
# ABRIR POWERPOINT
# ============================================================

ppt = Presentation(
    ARQUIVO_PPT_ENTRADA_12
)


# ============================================================
# SLIDES IDENTIFICADOS COMO DINÂMICOS
# ============================================================

SLIDES_DINAMICOS_12 = {
    13: "Painel Demográfico",
    14: "Investimento Financeiro CC",
    15: "Análise Financeira CC - Parte 1",
    16: "Análise Financeira CC - Parte 2",
    17: "Investimento Financeiro SC",
    18: "Análise Financeira SC - Parte 1",
    19: "Análise Financeira SC - Parte 2",
    21: "Rede Credenciada 1",
    22: "Rede Credenciada 2",
    23: "Rede Credenciada 3",
    24: "Rede Credenciada 4",
    25: "Rede Credenciada 5",
    26: "Rede Credenciada 6",
}


# ============================================================
# MAPEAR OBJETOS
# ============================================================

mapa_slides_12 = {}

total_objetos_12 = 0
total_textos_12 = 0
total_tabelas_12 = 0
total_graficos_12 = 0
total_imagens_12 = 0
total_vinculos_12 = 0


for numero_slide, slide in enumerate(
    ppt.slides,
    start=1
):

    titulo_mapeamento = (
        SLIDES_DINAMICOS_12.get(
            numero_slide,
            "Slide institucional ou estático"
        )
    )

    dados_slide = {
        "numero": numero_slide,
        "classificacao": titulo_mapeamento,
        "dinamico": (
            numero_slide
            in SLIDES_DINAMICOS_12
        ),
        "objetos": [],
    }

    for indice_shape, shape in enumerate(
        slide.shapes,
        start=1
    ):

        tipo_shape = nome_tipo_shape_12(
            shape
        )

        texto = extrair_texto_shape_12(
            shape
        )

        tabela = extrair_tabela_12(
            shape
        )

        grafico = extrair_grafico_12(
            shape
        )

        vinculos = extrair_vinculos_12(
            shape
        )

        registro = {
            "indice": indice_shape,
            "shape_id": shape.shape_id,
            "nome": shape.name,
            "tipo": tipo_shape,
            "texto": texto,
            "esquerda_cm": emu_para_cm_12(
                shape.left
            ),
            "topo_cm": emu_para_cm_12(
                shape.top
            ),
            "largura_cm": emu_para_cm_12(
                shape.width
            ),
            "altura_cm": emu_para_cm_12(
                shape.height
            ),
            "tabela": tabela,
            "grafico": grafico,
            "vinculos": vinculos,
        }

        dados_slide["objetos"].append(
            registro
        )

        total_objetos_12 += 1

        if texto:
            total_textos_12 += 1

        if tabela:
            total_tabelas_12 += 1

        if grafico:
            total_graficos_12 += 1

        if tipo_shape == "IMAGEM":
            total_imagens_12 += 1

        total_vinculos_12 += len(
            vinculos
        )

    mapa_slides_12[
        numero_slide
    ] = dados_slide


# ============================================================
# CRIAR RELATÓRIO
# ============================================================

with open(
    ARQUIVO_RELATORIO_12,
    "w",
    encoding="utf-8"
) as arquivo:

    arquivo.write(
        "=" * 70 + "\n"
    )

    arquivo.write(
        "APRESENTADOR 360 V2 - ETAPA 12\n"
    )

    arquivo.write(
        "MAPEAMENTO TÉCNICO DO POWERPOINT\n"
    )

    arquivo.write(
        "=" * 70 + "\n\n"
    )

    arquivo.write(
        "Arquivo analisado: "
        + ARQUIVO_PPT_ENTRADA_12
        + "\n"
    )

    arquivo.write(
        "Total de slides: "
        + str(len(ppt.slides))
        + "\n"
    )

    arquivo.write(
        "Slides dinâmicos identificados: "
        + str(len(SLIDES_DINAMICOS_12))
        + "\n"
    )

    arquivo.write(
        "Total de objetos: "
        + str(total_objetos_12)
        + "\n"
    )

    arquivo.write(
        "Objetos com texto: "
        + str(total_textos_12)
        + "\n"
    )

    arquivo.write(
        "Tabelas: "
        + str(total_tabelas_12)
        + "\n"
    )

    arquivo.write(
        "Gráficos: "
        + str(total_graficos_12)
        + "\n"
    )

    arquivo.write(
        "Imagens: "
        + str(total_imagens_12)
        + "\n"
    )

    arquivo.write(
        "Vínculos externos encontrados: "
        + str(total_vinculos_12)
        + "\n\n"
    )

    for numero_slide, dados_slide in (
        mapa_slides_12.items()
    ):

        arquivo.write(
            "-" * 70 + "\n"
        )

        arquivo.write(
            "SLIDE "
            + str(numero_slide)
            + "\n"
        )

        arquivo.write(
            "Classificação: "
            + dados_slide["classificacao"]
            + "\n"
        )

        arquivo.write(
            "Dinâmico: "
            + (
                "SIM"
                if dados_slide["dinamico"]
                else "NÃO"
            )
            + "\n"
        )

        arquivo.write(
            "Objetos: "
            + str(
                len(
                    dados_slide["objetos"]
                )
            )
            + "\n"
        )

        arquivo.write(
            "-" * 70 + "\n\n"
        )

        for objeto in dados_slide[
            "objetos"
        ]:

            arquivo.write(
                "Índice: "
                + str(objeto["indice"])
                + "\n"
            )

            arquivo.write(
                "Shape ID: "
                + str(objeto["shape_id"])
                + "\n"
            )

            arquivo.write(
                "Nome: "
                + objeto["nome"]
                + "\n"
            )

            arquivo.write(
                "Tipo: "
                + objeto["tipo"]
                + "\n"
            )

            arquivo.write(
                "Posição: "
                + "X="
                + str(objeto["esquerda_cm"])
                + " cm | Y="
                + str(objeto["topo_cm"])
                + " cm\n"
            )

            arquivo.write(
                "Tamanho: "
                + str(objeto["largura_cm"])
                + " x "
                + str(objeto["altura_cm"])
                + " cm\n"
            )

            if objeto["texto"]:

                arquivo.write(
                    "Texto: "
                    + objeto["texto"]
                    + "\n"
                )

            if objeto["tabela"]:

                arquivo.write(
                    "Tabela:\n"
                )

                for linha_tabela in objeto[
                    "tabela"
                ]:

                    arquivo.write(
                        "  "
                        + " | ".join(
                            linha_tabela
                        )
                        + "\n"
                    )

            if objeto["grafico"]:

                arquivo.write(
                    "Gráfico: "
                    + str(
                        objeto["grafico"]
                    )
                    + "\n"
                )

            if objeto["vinculos"]:

                arquivo.write(
                    "Vínculos:\n"
                )

                for vinculo in objeto[
                    "vinculos"
                ]:

                    arquivo.write(
                        "  "
                        + vinculo
                        + "\n"
                    )

            arquivo.write("\n")


# ============================================================
# ATUALIZAR ESTADO
# ============================================================

estado["arquivo_ppt_modelo"] = (
    ARQUIVO_PPT_ENTRADA_12
)

estado["relatorio_etapa_12"] = (
    ARQUIVO_RELATORIO_12
)

estado["mapa_slides_etapa_12"] = (
    mapa_slides_12
)

estado["slides_dinamicos"] = (
    dict(
        SLIDES_DINAMICOS_12
    )
)


# ============================================================
# RESUMO
# ============================================================

print("\n" + "=" * 70)
print("RESUMO DA ETAPA 12")
print("=" * 70)

print(
    "PowerPoint analisado:",
    ARQUIVO_PPT_ENTRADA_12
)

print(
    "Total de slides:",
    len(ppt.slides)
)

print(
    "Slides dinâmicos identificados:",
    len(SLIDES_DINAMICOS_12)
)

print(
    "Total de objetos:",
    total_objetos_12
)

print(
    "Objetos com texto:",
    total_textos_12
)

print(
    "Tabelas encontradas:",
    total_tabelas_12
)

print(
    "Gráficos encontrados:",
    total_graficos_12
)

print(
    "Imagens encontradas:",
    total_imagens_12
)

print(
    "Vínculos externos encontrados:",
    total_vinculos_12
)

print(
    "Relatório criado:",
    ARQUIVO_RELATORIO_12
)

print(
    "\n✅ ETAPA 12 CONCLUÍDA COM SUCESSO"
)

print("=" * 70)

ETAPA 12 - MAPEANDO OBJETOS DO POWERPOINT

RESUMO DA ETAPA 12
PowerPoint analisado: Mezzo_Apresentação Saúde_2026_07_v1.pptx
Total de slides: 28
Slides dinâmicos identificados: 13
Total de objetos: 407
Objetos com texto: 181
Tabelas encontradas: 16
Gráficos encontrados: 4
Imagens encontradas: 92
Vínculos externos encontrados: 0
Relatório criado: Apresentador360_V2_Mapeamento_PPT_Etapa12.txt

✅ ETAPA 12 CONCLUÍDA COM SUCESSO


In [53]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 13 - VERSÃO CORRIGIDA
# ATUALIZAR O SLIDE 13 - PAINEL DEMOGRÁFICO
# PRESERVANDO GRÁFICOS COM VÍNCULO EXTERNO
# ============================================================

import os
import re
import unicodedata

from pptx import Presentation


print("=" * 70)
print("ETAPA 13 - ATUALIZANDO O PAINEL DEMOGRÁFICO")
print("=" * 70)


# ============================================================
# CONFIGURAÇÃO
# ============================================================

if "estado" not in globals():
    raise NameError(
        "O estado do projeto não existe. "
        "Execute as etapas anteriores."
    )


ARQUIVO_PPT_ENTRADA_13 = estado.get(
    "arquivo_ppt_modelo",
    "Mezzo_Apresentação Saúde_2026_07_v1.pptx"
)

ARQUIVO_PPT_SAIDA_13 = (
    "Apresentador360_V2_Etapa13_Painel_Demografico.pptx"
)

ARQUIVO_RELATORIO_13 = (
    "Apresentador360_V2_Relatorio_Etapa13.txt"
)

NUMERO_SLIDE_13 = 13


# ============================================================
# VALIDAR ESTADO
# ============================================================

CHAVES_OBRIGATORIAS_13 = [
    "elegibilidade",
    "faixa_etaria",
    "produtos_atuais",
]


chaves_faltantes_13 = [
    chave
    for chave in CHAVES_OBRIGATORIAS_13
    if chave not in estado
]


if chaves_faltantes_13:
    raise ValueError(
        "Informações ausentes no estado: "
        + ", ".join(chaves_faltantes_13)
    )


if not os.path.exists(
    ARQUIVO_PPT_ENTRADA_13
):
    raise FileNotFoundError(
        "O PowerPoint não foi encontrado: "
        + ARQUIVO_PPT_ENTRADA_13
    )


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def limpar_texto_13(valor):

    if valor is None:
        return ""

    return " ".join(
        str(valor).replace("\n", " ").split()
    ).strip()


def normalizar_13(valor):

    texto = limpar_texto_13(valor)

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(
            caractere
        )
    )

    texto = texto.upper()

    texto = re.sub(
        r"[^A-Z0-9%]+",
        " ",
        texto
    )

    return " ".join(
        texto.split()
    )


def percentual_inteiro_13(
    quantidade,
    total
):

    if not total:
        return 0

    return round(
        quantidade
        / total
        * 100
    )


def substituir_texto_13(
    shape,
    novo_texto
):
    """
    Substitui o texto tentando preservar
    a formatação original da primeira execução.
    """

    if not getattr(
        shape,
        "has_text_frame",
        False
    ):
        return False

    text_frame = shape.text_frame

    if not text_frame.paragraphs:
        return False

    primeiro_paragrafo = (
        text_frame.paragraphs[0]
    )

    primeiro_run = None

    for paragrafo in text_frame.paragraphs:

        if paragrafo.runs:

            primeiro_run = (
                paragrafo.runs[0]
            )

            break

    estilo = {}

    if primeiro_run is not None:

        try:
            estilo["font_name"] = (
                primeiro_run.font.name
            )
        except Exception:
            estilo["font_name"] = None

        try:
            estilo["font_size"] = (
                primeiro_run.font.size
            )
        except Exception:
            estilo["font_size"] = None

        try:
            estilo["bold"] = (
                primeiro_run.font.bold
            )
        except Exception:
            estilo["bold"] = None

        try:
            estilo["italic"] = (
                primeiro_run.font.italic
            )
        except Exception:
            estilo["italic"] = None

        try:
            estilo["color"] = (
                primeiro_run.font.color.rgb
            )
        except Exception:
            estilo["color"] = None

    text_frame.clear()

    paragrafo = text_frame.paragraphs[0]

    novo_run = paragrafo.add_run()

    novo_run.text = str(
        novo_texto
    )

    if estilo.get("font_name"):

        novo_run.font.name = estilo[
            "font_name"
        ]

    if estilo.get("font_size"):

        novo_run.font.size = estilo[
            "font_size"
        ]

    if estilo.get("bold") is not None:

        novo_run.font.bold = estilo[
            "bold"
        ]

    if estilo.get("italic") is not None:

        novo_run.font.italic = estilo[
            "italic"
        ]

    if estilo.get("color") is not None:

        try:
            novo_run.font.color.rgb = (
                estilo["color"]
            )
        except Exception:
            pass

    return True


def categorias_do_grafico_13(
    chart
):

    categorias = []

    try:

        for categoria in (
            chart.plots[0].categories
        ):

            categorias.append(
                limpar_texto_13(
                    categoria.label
                )
            )

    except Exception:
        pass

    return categorias


def classificar_grafico_13(
    categorias
):

    texto = normalizar_13(
        " ".join(categorias)
    )

    if (
        "FEMININO" in texto
        or "MASCULINO" in texto
    ):
        return "GÊNERO"

    if (
        "FUNCIONARIO" in texto
        or "DEPENDENTE" in texto
        or "TITULAR" in texto
    ):
        return "TITULARIDADE"

    faixas = [
        "00 18",
        "19 23",
        "24 28",
        "29 33",
        "34 38",
        "39 43",
        "44 48",
        "49 53",
        "54 58",
        "59",
    ]

    quantidade_faixas = sum(
        1
        for faixa in faixas
        if faixa in texto
    )

    if quantidade_faixas >= 3:
        return "FAIXA ETÁRIA"

    if len(categorias) >= 3:
        return "DISTRIBUIÇÃO DE PLANOS"

    return "NÃO IDENTIFICADO"


def grafico_tem_vinculo_externo_13(
    chart
):
    """
    Verifica se o gráfico possui relacionamento
    externo com uma planilha.
    """

    try:

        relacionamentos = (
            chart.part.rels.values()
        )

        for relacionamento in relacionamentos:

            if relacionamento.is_external:
                return True

    except Exception:
        pass

    return False


# ============================================================
# PREPARAR DADOS DO ESTADO
# ============================================================

elegibilidade_13 = estado[
    "elegibilidade"
]

faixa_etaria_13 = estado[
    "faixa_etaria"
]

produtos_atuais_13 = estado[
    "produtos_atuais"
]


total_vidas_13 = int(
    elegibilidade_13.get(
        "total_vidas",
        0
    )
    or 0
)

feminino_13 = int(
    elegibilidade_13.get(
        "feminino",
        0
    )
    or 0
)

masculino_13 = int(
    elegibilidade_13.get(
        "masculino",
        0
    )
    or 0
)

funcionarios_13 = int(
    elegibilidade_13.get(
        "funcionarios",
        0
    )
    or 0
)

dependentes_13 = int(
    elegibilidade_13.get(
        "dependentes",
        0
    )
    or 0
)


percentual_feminino_13 = (
    percentual_inteiro_13(
        feminino_13,
        total_vidas_13
    )
)

percentual_masculino_13 = (
    percentual_inteiro_13(
        masculino_13,
        total_vidas_13
    )
)


# ============================================================
# PREPARAR FAIXAS ETÁRIAS
# ============================================================

ORDEM_FAIXAS_13 = [
    "00 - 18",
    "19 - 23",
    "24 - 28",
    "29 - 33",
    "34 - 38",
    "39 - 43",
    "44 - 48",
    "49 - 53",
    "54 - 58",
    "59+",
]


valores_faixas_13 = {}


for faixa in ORDEM_FAIXAS_13:

    dados_faixa = faixa_etaria_13.get(
        faixa,
        {}
    )

    funcionarios_faixa = int(
        dados_faixa.get(
            "funcionarios",
            0
        )
        or 0
    )

    dependentes_faixa = int(
        dados_faixa.get(
            "dependentes",
            0
        )
        or 0
    )

    valores_faixas_13[
        faixa
    ] = (
        funcionarios_faixa
        + dependentes_faixa
    )


# ============================================================
# PREPARAR PRODUTOS
# ============================================================

distribuicao_produtos_13 = {}


for produto in produtos_atuais_13:

    plano = limpar_texto_13(
        produto.get(
            "plano",
            ""
        )
    )

    vidas = int(
        produto.get(
            "vidas",
            0
        )
        or 0
    )

    if plano and vidas > 0:

        distribuicao_produtos_13[
            plano
        ] = vidas


# ============================================================
# ABRIR POWERPOINT
# ============================================================

ppt = Presentation(
    ARQUIVO_PPT_ENTRADA_13
)


if len(ppt.slides) < NUMERO_SLIDE_13:

    raise ValueError(
        "O PowerPoint possui menos de "
        "13 slides."
    )


quantidade_slides_original_13 = len(
    ppt.slides
)


slide_13 = ppt.slides[
    NUMERO_SLIDE_13 - 1
]


# ============================================================
# ATUALIZAR TEXTOS DO SLIDE 13
# ============================================================

formas_percentuais_13 = []

textos_atualizados_13 = []


for shape in slide_13.shapes:

    if not getattr(
        shape,
        "has_text_frame",
        False
    ):
        continue

    texto_original = limpar_texto_13(
        shape.text
    )

    if not texto_original:
        continue

    texto_normalizado = normalizar_13(
        texto_original
    )


    # --------------------------------------------------------
    # CAIXAS DE PERCENTUAL
    # --------------------------------------------------------

    if re.fullmatch(
        r"[0-9]{1,3}%",
        texto_original
    ):

        formas_percentuais_13.append(
            shape
        )

        continue


    # --------------------------------------------------------
    # FEMININO
    # --------------------------------------------------------

    if "FEMININO" in texto_normalizado:

        novo_texto = (
            "Feminino ("
            + str(feminino_13)
            + ")"
        )

        if substituir_texto_13(
            shape,
            novo_texto
        ):

            textos_atualizados_13.append(
                texto_original
                + " -> "
                + novo_texto
            )

        continue


    # --------------------------------------------------------
    # MASCULINO
    # --------------------------------------------------------

    if "MASCULINO" in texto_normalizado:

        novo_texto = (
            "Masculino\n("
            + str(masculino_13)
            + ")"
        )

        if substituir_texto_13(
            shape,
            novo_texto
        ):

            textos_atualizados_13.append(
                texto_original
                + " -> "
                + novo_texto
            )


# ============================================================
# ATUALIZAR PERCENTUAIS
# ============================================================

formas_percentuais_13 = sorted(
    formas_percentuais_13,
    key=lambda objeto: (
        objeto.top,
        objeto.left
    )
)

if len(formas_percentuais_13) >= 1:

    substituir_texto_13(
        formas_percentuais_13[0],
        str(percentual_masculino_13) + "%"
    )

if len(formas_percentuais_13) >= 2:

    substituir_texto_13(
        formas_percentuais_13[1],
        str(percentual_feminino_13) + "%"
    )



# ============================================================
# MAPEAR E PRESERVAR OS GRÁFICOS
# ============================================================
#
# Os gráficos do modelo possuem vínculo externo.
# O python-pptx não consegue substituir esses dados
# diretamente sem quebrar o relacionamento.
#
# Nesta etapa:
# - os gráficos permanecem intactos;
# - o PowerPoint é salvo normalmente;
# - as classificações ficam registradas;
# - nenhum replace_data() é executado.
# ============================================================

graficos_encontrados_13 = 0

graficos_com_vinculo_13 = 0

graficos_preservados_13 = []


for indice_shape, shape in enumerate(
    slide_13.shapes,
    start=1
):

    if not getattr(
        shape,
        "has_chart",
        False
    ):
        continue

    graficos_encontrados_13 += 1

    chart = shape.chart

    categorias = (
        categorias_do_grafico_13(
            chart
        )
    )

    classificacao = (
        classificar_grafico_13(
            categorias
        )
    )

    possui_vinculo_externo = (
        grafico_tem_vinculo_externo_13(
            chart
        )
    )

    if possui_vinculo_externo:
        graficos_com_vinculo_13 += 1
    graficos_preservados_13.append({
        "shape": indice_shape,
        "nome": shape.name,
        "classificacao": classificacao,
        "categorias": categorias,
        "vinculo_externo": possui_vinculo_externo,
        "status": "PRESERVADO",
    })


# ============================================================
# SALVAR NOVO POWERPOINT
# ============================================================

if os.path.exists(
    ARQUIVO_PPT_SAIDA_13
):
    os.remove(
        ARQUIVO_PPT_SAIDA_13
    )


ppt.save(
    ARQUIVO_PPT_SAIDA_13
)


# ============================================================
# VALIDAR O ARQUIVO GERADO
# ============================================================

ppt_validacao_13 = Presentation(
    ARQUIVO_PPT_SAIDA_13
)


quantidade_slides_final_13 = len(
    ppt_validacao_13.slides
)


if (
    quantidade_slides_final_13
    != quantidade_slides_original_13
):

    raise RuntimeError(
        "A quantidade de slides foi alterada."
    )


slide_validacao_13 = (
    ppt_validacao_13.slides[12]
)


textos_validacao_13 = []


for shape in slide_validacao_13.shapes:

    if getattr(
        shape,
        "has_text_frame",
        False
    ):

        texto = limpar_texto_13(
            shape.text
        )

        if texto:

            textos_validacao_13.append(
                texto
            )


texto_validacao_completo_13 = (
    " | ".join(
        textos_validacao_13
    )
)


if (
    str(feminino_13)
    not in texto_validacao_completo_13
):

    raise RuntimeError(
        "O valor feminino não foi localizado "
        "no PowerPoint gerado."
    )


if (
    str(masculino_13)
    not in texto_validacao_completo_13
):

    raise RuntimeError(
        "O valor masculino não foi localizado "
        "no PowerPoint gerado."
    )


# ============================================================
# CRIAR RELATÓRIO
# ============================================================

with open(
    ARQUIVO_RELATORIO_13,
    "w",
    encoding="utf-8"
) as arquivo:

    arquivo.write(
        "=" * 70 + "\n"
    )

    arquivo.write(
        "APRESENTADOR 360 V2 - ETAPA 13\n"
    )

    arquivo.write(
        "PAINEL DEMOGRÁFICO - SLIDE 13\n"
    )

    arquivo.write(
        "=" * 70 + "\n\n"
    )

    arquivo.write(
        "PowerPoint de entrada: "
        + ARQUIVO_PPT_ENTRADA_13
        + "\n"
    )

    arquivo.write(
        "PowerPoint criado: "
        + ARQUIVO_PPT_SAIDA_13
        + "\n"
    )

    arquivo.write(
        "Slide atualizado: 13\n"
    )

    arquivo.write(
        "Quantidade de slides preservada: SIM\n"
    )

    arquivo.write(
        "Slides 1 a 12 preservados: SIM\n"
    )

    arquivo.write(
        "Slides 14 a 28 preservados: SIM\n\n"
    )

    arquivo.write(
        "Total de vidas: "
        + str(total_vidas_13)
        + "\n"
    )

    arquivo.write(
        "Feminino: "
        + str(feminino_13)
        + " | "
        + str(percentual_feminino_13)
        + "%\n"
    )

    arquivo.write(
        "Masculino: "
        + str(masculino_13)
        + " | "
        + str(percentual_masculino_13)
        + "%\n"
    )

    arquivo.write(
        "Funcionários: "
        + str(funcionarios_13)
        + "\n"
    )

    arquivo.write(
        "Dependentes: "
        + str(dependentes_13)
        + "\n"
    )

    arquivo.write(
        "Produtos atuais: "
        + str(
            len(
                distribuicao_produtos_13
            )
        )
        + "\n"
    )

    arquivo.write(
        "Gráficos encontrados: "
        + str(graficos_encontrados_13)
        + "\n"
    )

    arquivo.write(
        "Gráficos com vínculo externo: "
        + str(graficos_com_vinculo_13)
        + "\n"
    )

    arquivo.write(
        "Gráficos preservados: "
        + str(
            len(
                graficos_preservados_13
            )
        )
        + "\n\n"
    )

    arquivo.write(
        "-" * 70 + "\n"
    )

    arquivo.write(
        "TEXTOS ATUALIZADOS\n"
    )

    arquivo.write(
        "-" * 70 + "\n"
    )

    if textos_atualizados_13:

        for texto in textos_atualizados_13:

            arquivo.write(
                "- "
                + texto
                + "\n"
            )

    else:

        arquivo.write(
            "- Nenhum texto atualizado.\n"
        )

    arquivo.write(
        "\n"
        + "-" * 70
        + "\n"
    )

    arquivo.write(
        "GRÁFICOS PRESERVADOS\n"
    )

    arquivo.write(
        "-" * 70 + "\n"
    )

    for grafico in graficos_preservados_13:

        arquivo.write(
            "Shape: "
            + str(grafico["shape"])
            + "\n"
        )

        arquivo.write(
            "Nome: "
            + str(grafico["nome"])
            + "\n"
        )

        arquivo.write(
            "Classificação: "
            + grafico["classificacao"]
            + "\n"
        )

        arquivo.write(
            "Vínculo externo: "
            + (
                "SIM"
                if grafico["vinculo_externo"]
                else "NÃO"
            )
            + "\n"
        )

        arquivo.write(
            "Status: "
            + grafico["status"]
            + "\n\n"
        )


# ============================================================
# ATUALIZAR ESTADO
# ============================================================

estado["arquivo_ppt_etapa_13"] = (
    ARQUIVO_PPT_SAIDA_13
)

estado["relatorio_etapa_13"] = (
    ARQUIVO_RELATORIO_13
)

estado["dados_demograficos_etapa_13"] = {
    "total_vidas": total_vidas_13,
    "feminino": feminino_13,
    "masculino": masculino_13,
    "percentual_feminino": percentual_feminino_13,
    "percentual_masculino": percentual_masculino_13,
    "funcionarios": funcionarios_13,
    "dependentes": dependentes_13,
    "faixas": valores_faixas_13,
    "produtos": distribuicao_produtos_13,
}

estado["graficos_preservados_etapa_13"] = (
    graficos_preservados_13
)


# ============================================================
# RESUMO
# ============================================================

print("\n" + "=" * 70)
print("RESUMO DA ETAPA 13")
print("=" * 70)

print(
    "PowerPoint criado:",
    ARQUIVO_PPT_SAIDA_13
)

print(
    "Slide atualizado:",
    NUMERO_SLIDE_13
)

print(
    "Total de vidas:",
    total_vidas_13
)

print(
    "Feminino:",
    feminino_13,
    "|",
    str(percentual_feminino_13) + "%"
)

print(
    "Masculino:",
    masculino_13,
    "|",
    str(percentual_masculino_13) + "%"
)

print(
    "Funcionários:",
    funcionarios_13
)

print(
    "Dependentes:",
    dependentes_13
)

print(
    "Gráficos encontrados:",
    graficos_encontrados_13
)

print(
    "Gráficos com vínculo externo:",
    graficos_com_vinculo_13
)

print(
    "Gráficos preservados:",
    len(graficos_preservados_13)
)

print(
    "Quantidade de slides:",
    quantidade_slides_final_13
)

print(
    "Relatório criado:",
    ARQUIVO_RELATORIO_13
)

print(
    "\n✅ ETAPA 13 CONCLUÍDA COM SUCESSO"
)

print("=" * 70)

ETAPA 13 - ATUALIZANDO O PAINEL DEMOGRÁFICO

RESUMO DA ETAPA 13
PowerPoint criado: Apresentador360_V2_Etapa13_Painel_Demografico.pptx
Slide atualizado: 13
Total de vidas: 24
Feminino: 13 | 54%
Masculino: 11 | 46%
Funcionários: 19
Dependentes: 5
Gráficos encontrados: 4
Gráficos com vínculo externo: 4
Gráficos preservados: 4
Quantidade de slides: 28
Relatório criado: Apresentador360_V2_Relatorio_Etapa13.txt

✅ ETAPA 13 CONCLUÍDA COM SUCESSO


In [54]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 14
# ATUALIZAR SLIDES 14 E 17
# INVESTIMENTO FINANCEIRO CC E SC
# ============================================================

import os
import re

from openpyxl import load_workbook
from pptx import Presentation


print("=" * 70)
print("ETAPA 14 - ATUALIZANDO INVESTIMENTO FINANCEIRO")
print("=" * 70)


# ============================================================
# CONFIGURAÇÃO
# ============================================================

if "estado" not in globals():
    raise NameError(
        "O estado do projeto não existe. "
        "Execute as etapas anteriores."
    )


ARQUIVO_EXCEL_14 = estado.get(
    "arquivo_etapa_10",
    "Apresentador360_V2_Etapa10.xlsx"
)

ARQUIVO_PPT_ENTRADA_14 = estado.get(
    "arquivo_ppt_etapa_13",
    "Apresentador360_V2_Etapa13_Painel_Demografico.pptx"
)

ARQUIVO_PPT_SAIDA_14 = (
    "Apresentador360_V2_Etapa14_Financeiro.pptx"
)

ARQUIVO_RELATORIO_14 = (
    "Apresentador360_V2_Relatorio_Etapa14.txt"
)


if not os.path.exists(ARQUIVO_EXCEL_14):
    raise FileNotFoundError(
        "O Excel da Etapa 10 não foi encontrado: "
        + ARQUIVO_EXCEL_14
    )


if not os.path.exists(ARQUIVO_PPT_ENTRADA_14):
    raise FileNotFoundError(
        "O PowerPoint da Etapa 13 não foi encontrado: "
        + ARQUIVO_PPT_ENTRADA_14
    )


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def limpar_texto_14(valor):

    if valor is None:
        return ""

    return " ".join(
        str(valor).replace("\n", " ").split()
    ).strip()


def numero_14(valor):

    try:
        return float(valor or 0)

    except (TypeError, ValueError):
        return 0.0


def formatar_moeda_slide_14(valor):

    valor_arredondado = round(
        abs(numero_14(valor))
    )

    return (
        "R$"
        + f"{valor_arredondado:,}"
        .replace(",", ".")
    )


def formatar_reducao_14(valor):

    valor_arredondado = round(
        abs(numero_14(valor))
    )

    return (
        "Redução Mensal\n- R$"
        + f"{valor_arredondado:,}"
        .replace(",", ".")
    )


def formatar_percentual_14(valor):

    percentual = round(
        numero_14(valor) * 100
    )

    if percentual <= 0:

        return (
            "- "
            + str(abs(percentual))
            + "%"
        )

    return (
        "+ "
        + str(percentual)
        + "%"
    )


def substituir_texto_preservando_estilo_14(
    shape,
    novo_texto
):
    """
    Substitui o conteúdo preservando o estilo
    do primeiro trecho de texto da forma.
    """

    if not getattr(
        shape,
        "has_text_frame",
        False
    ):
        return False

    text_frame = shape.text_frame

    estilo = {
        "nome": None,
        "tamanho": None,
        "negrito": None,
        "italico": None,
        "cor": None,
    }

    for paragrafo in text_frame.paragraphs:

        if not paragrafo.runs:
            continue

        run_original = paragrafo.runs[0]

        try:
            estilo["nome"] = run_original.font.name
        except Exception:
            pass

        try:
            estilo["tamanho"] = run_original.font.size
        except Exception:
            pass

        try:
            estilo["negrito"] = run_original.font.bold
        except Exception:
            pass

        try:
            estilo["italico"] = run_original.font.italic
        except Exception:
            pass

        try:
            estilo["cor"] = run_original.font.color.rgb
        except Exception:
            pass

        break

    text_frame.clear()

    paragrafo = text_frame.paragraphs[0]

    novo_run = paragrafo.add_run()

    novo_run.text = str(novo_texto)

    if estilo["nome"]:
        novo_run.font.name = estilo["nome"]

    if estilo["tamanho"]:
        novo_run.font.size = estilo["tamanho"]

    if estilo["negrito"] is not None:
        novo_run.font.bold = estilo["negrito"]

    if estilo["italico"] is not None:
        novo_run.font.italic = estilo["italico"]

    if estilo["cor"] is not None:

        try:
            novo_run.font.color.rgb = estilo["cor"]
        except Exception:
            pass

    return True


def localizar_formas_por_padrao_14(
    slide,
    padrao
):

    encontrados = []

    for shape in slide.shapes:

        if not getattr(
            shape,
            "has_text_frame",
            False
        ):
            continue

        texto = limpar_texto_14(
            shape.text
        )

        if re.search(
            padrao,
            texto,
            flags=re.IGNORECASE
        ):

            encontrados.append(shape)

    return encontrados


def atualizar_slide_investimento_14(
    slide,
    valor_atual,
    valor_reajustado,
    valores_cenarios,
    total_vidas
):
    """
    Atualiza:
    - valores monetários;
    - quantidade de vidas;
    - percentuais;
    - reduções mensais.

    Preserva todo o restante do slide.
    """

    atualizacoes = []

    formas_moeda = localizar_formas_por_padrao_14(
        slide,
        r"^R\$\s*[\d\.\,]+$"
    )

    formas_percentual = localizar_formas_por_padrao_14(
        slide,
        r"^[\+\-]?\s*\d+\s*%$"
    )

    formas_reducao = localizar_formas_por_padrao_14(
        slide,
        r"REDU[CÇ][AÃ]O\s+MENSAL"
    )

    formas_vidas = localizar_formas_por_padrao_14(
        slide,
        r"TOTAL\s+DE\s+VIDAS"
    )


    # Ordenar na sequência visual do slide.
    formas_moeda = sorted(
        formas_moeda,
        key=lambda shape: (
            shape.top,
            shape.left
        )
    )

    formas_percentual = sorted(
        formas_percentual,
        key=lambda shape: (
            shape.top,
            shape.left
        )
    )

    formas_reducao = sorted(
        formas_reducao,
        key=lambda shape: (
            shape.left,
            shape.top
        )
    )


    valores_moeda = [
        valor_atual,
        valor_reajustado,
        *valores_cenarios,
    ]


    for shape, valor in zip(
        formas_moeda,
        valores_moeda
    ):

        novo_texto = formatar_moeda_slide_14(
            valor
        )

        substituir_texto_preservando_estilo_14(
            shape,
            novo_texto
        )

        atualizacoes.append(
            "Moeda: " + novo_texto
        )


    for shape in formas_vidas:

        novo_texto = (
            "Total de Vidas: "
            + str(total_vidas)
        )

        substituir_texto_preservando_estilo_14(
            shape,
            novo_texto
        )


    percentuais = [
        (
            valor / valor_reajustado - 1
            if valor_reajustado
            else 0
        )
        for valor in valores_cenarios
    ]


    for shape, percentual in zip(
        formas_percentual,
        percentuais
    ):

        novo_texto = formatar_percentual_14(
            percentual
        )

        substituir_texto_preservando_estilo_14(
            shape,
            novo_texto
        )

        atualizacoes.append(
            "Percentual: " + novo_texto
        )


    reducoes = [
        valor_reajustado - valor
        for valor in reversed(
            valores_cenarios
        )
    ]


    for shape, reducao in zip(
        formas_reducao,
        reducoes
    ):

        novo_texto = formatar_reducao_14(
            reducao
        )

        substituir_texto_preservando_estilo_14(
            shape,
            novo_texto
        )

        atualizacoes.append(
            novo_texto.replace("\n", " ")
        )


    return {
        "formas_moeda": len(formas_moeda),
        "formas_percentual": len(formas_percentual),
        "formas_reducao": len(formas_reducao),
        "formas_vidas": len(formas_vidas),
        "atualizacoes": atualizacoes,
    }


# ============================================================
# ABRIR EXCEL
# ============================================================

wb = load_workbook(
    ARQUIVO_EXCEL_14,
    data_only=False
)


for aba_obrigatoria in [
    "Planos",
    "Produtos Atuais",
    "Análise Financeira CC",
    "Análise Financeira SC",
]:

    if aba_obrigatoria not in wb.sheetnames:

        wb.close()

        raise ValueError(
            "A aba não foi encontrada: "
            + aba_obrigatoria
        )


ws_planos = wb["Planos"]

ws_produtos = wb[
    "Produtos Atuais"
]

ws_analise_cc = wb[
    "Análise Financeira CC"
]


# ============================================================
# PRODUTOS E VIDAS
# ============================================================

vidas_produtos_14 = []

custos_atuais_14 = []


for indice in range(10):

    linha_produtos = 2 + indice

    coluna_planos = 3 + indice

    vidas = numero_14(
        ws_produtos.cell(
            linha_produtos,
            2
        ).value
    )

    custo_atual = numero_14(
        ws_planos.cell(
            11,
            coluna_planos
        ).value
    )

    vidas_produtos_14.append(
        vidas
    )

    custos_atuais_14.append(
        custo_atual
    )


total_vidas_14 = round(
    sum(vidas_produtos_14)
)


valor_atual_14 = sum(
    vidas * custo
    for vidas, custo in zip(
        vidas_produtos_14,
        custos_atuais_14
    )
)


# ============================================================
# CALCULAR REAJUSTE
# ============================================================

fatores_reajuste_14 = []


linhas_custos_analise_14 = [
    5,
    10,
    15,
    20,
    25,
    30,
    35,
    40,
    45,
    50,
]


for custo_atual, linha_custo in zip(
    custos_atuais_14,
    linhas_custos_analise_14
):

    formula = limpar_texto_14(
        ws_analise_cc.cell(
            linha_custo,
            3
        ).value
    )

    fator = 1.0

    referencia = re.search(
        r"\*([A-Z]+)(\d+)",
        formula
    )

    if referencia:

        coluna_fator = referencia.group(1)

        linha_fator = int(
            referencia.group(2)
        )

        fator = numero_14(
            ws_analise_cc[
                coluna_fator
                + str(linha_fator)
            ].value
        )

    if fator <= 0:
        fator = 1.0

    fatores_reajuste_14.append(
        fator
    )


valor_reajustado_14 = sum(
    vidas * custo * fator
    for vidas, custo, fator in zip(
        vidas_produtos_14,
        custos_atuais_14,
        fatores_reajuste_14
    )
)


# ============================================================
# CALCULAR CENÁRIOS CC E SC
# ============================================================

quantidade_cenarios_14 = min(
    len(
        estado.get(
            "cenarios_escolhidos",
            []
        )
    ),
    3
)


linhas_cc_14 = [
    13,
    16,
    19,
    22,
]


linhas_sc_14 = [
    14,
    17,
    20,
    23,
]


def calcular_totais_cenarios_14(
    linhas_custos
):

    totais = []

    for linha_custo in linhas_custos[
        :quantidade_cenarios_14
    ]:

        total = 0.0

        for indice_produto in range(10):

            coluna_produto = (
                3 + indice_produto
            )

            custo = numero_14(
                ws_planos.cell(
                    linha_custo,
                    coluna_produto
                ).value
            )

            vidas = vidas_produtos_14[
                indice_produto
            ]

            total += vidas * custo

        totais.append(total)

    while len(totais) < 3:
        totais.append(0.0)

    return totais


totais_cc_14 = calcular_totais_cenarios_14(
    linhas_cc_14
)

totais_sc_14 = calcular_totais_cenarios_14(
    linhas_sc_14
)


wb.close()


# ============================================================
# ABRIR POWERPOINT DA ETAPA 13
# ============================================================

ppt = Presentation(
    ARQUIVO_PPT_ENTRADA_14
)


if len(ppt.slides) != 28:

    raise ValueError(
        "O PowerPoint deve possuir 28 slides."
    )


slide_14 = ppt.slides[13]

slide_17 = ppt.slides[16]


# ============================================================
# ATUALIZAR SLIDE 14 - CC
# ============================================================

resultado_slide_14 = (
    atualizar_slide_investimento_14(
        slide=slide_14,
        valor_atual=valor_atual_14,
        valor_reajustado=valor_reajustado_14,
        valores_cenarios=totais_cc_14,
        total_vidas=total_vidas_14,
    )
)


# ============================================================
# ATUALIZAR SLIDE 17 - SC
# ============================================================

resultado_slide_17 = (
    atualizar_slide_investimento_14(
        slide=slide_17,
        valor_atual=valor_atual_14,
        valor_reajustado=valor_reajustado_14,
        valores_cenarios=totais_sc_14,
        total_vidas=total_vidas_14,
    )
)


# ============================================================
# SALVAR POWERPOINT
# ============================================================

if os.path.exists(
    ARQUIVO_PPT_SAIDA_14
):

    os.remove(
        ARQUIVO_PPT_SAIDA_14
    )


ppt.save(
    ARQUIVO_PPT_SAIDA_14
)


# ============================================================
# VALIDAR POWERPOINT GERADO
# ============================================================

ppt_validacao_14 = Presentation(
    ARQUIVO_PPT_SAIDA_14
)


if len(ppt_validacao_14.slides) != 28:

    raise RuntimeError(
        "A quantidade de slides foi alterada."
    )


# ============================================================
# CRIAR RELATÓRIO
# ============================================================

with open(
    ARQUIVO_RELATORIO_14,
    "w",
    encoding="utf-8"
) as arquivo:

    arquivo.write(
        "=" * 70 + "\n"
    )

    arquivo.write(
        "APRESENTADOR 360 V2 - ETAPA 14\n"
    )

    arquivo.write(
        "INVESTIMENTO FINANCEIRO CC E SC\n"
    )

    arquivo.write(
        "=" * 70 + "\n\n"
    )

    arquivo.write(
        "PowerPoint criado: "
        + ARQUIVO_PPT_SAIDA_14
        + "\n"
    )

    arquivo.write(
        "Slide 14 atualizado: SIM\n"
    )

    arquivo.write(
        "Slide 17 atualizado: SIM\n"
    )

    arquivo.write(
        "Slides 20 a 28 preservados: SIM\n\n"
    )

    arquivo.write(
        "Total de vidas: "
        + str(total_vidas_14)
        + "\n"
    )

    arquivo.write(
        "Valor atual: "
        + formatar_moeda_slide_14(
            valor_atual_14
        )
        + "\n"
    )

    arquivo.write(
        "Valor reajustado: "
        + formatar_moeda_slide_14(
            valor_reajustado_14
        )
        + "\n\n"
    )

    arquivo.write(
        "CENÁRIOS CC:\n"
    )

    for indice, valor in enumerate(
        totais_cc_14,
        start=1
    ):

        arquivo.write(
            "Cenário "
            + str(indice)
            + ": "
            + formatar_moeda_slide_14(
                valor
            )
            + "\n"
        )

    arquivo.write(
        "\nCENÁRIOS SC:\n"
    )

    for indice, valor in enumerate(
        totais_sc_14,
        start=1
    ):

        arquivo.write(
            "Cenário "
            + str(indice)
            + ": "
            + formatar_moeda_slide_14(
                valor
            )
            + "\n"
        )


# ============================================================
# ATUALIZAR ESTADO
# ============================================================

estado["arquivo_ppt_etapa_14"] = (
    ARQUIVO_PPT_SAIDA_14
)

estado["relatorio_etapa_14"] = (
    ARQUIVO_RELATORIO_14
)

estado["resumo_financeiro_etapa_14"] = {
    "total_vidas": total_vidas_14,
    "valor_atual": valor_atual_14,
    "valor_reajustado": valor_reajustado_14,
    "cenarios_cc": totais_cc_14,
    "cenarios_sc": totais_sc_14,
}


# ============================================================
# RESUMO
# ============================================================

print("\n" + "=" * 70)
print("RESUMO DA ETAPA 14")
print("=" * 70)

print(
    "PowerPoint criado:",
    ARQUIVO_PPT_SAIDA_14
)

print(
    "Slides atualizados:",
    "14 e 17"
)

print(
    "Total de vidas:",
    total_vidas_14
)

print(
    "Valor atual:",
    formatar_moeda_slide_14(
        valor_atual_14
    )
)

print(
    "Valor reajustado:",
    formatar_moeda_slide_14(
        valor_reajustado_14
    )
)

print(
    "Cenários CC:",
    " | ".join(
        formatar_moeda_slide_14(valor)
        for valor in totais_cc_14
    )
)

print(
    "Cenários SC:",
    " | ".join(
        formatar_moeda_slide_14(valor)
        for valor in totais_sc_14
    )
)

print(
    "Quantidade de slides:",
    len(ppt_validacao_14.slides)
)

print(
    "Relatório criado:",
    ARQUIVO_RELATORIO_14
)

print(
    "\n✅ ETAPA 14 CONCLUÍDA COM SUCESSO"
)

print("=" * 70)

ETAPA 14 - ATUALIZANDO INVESTIMENTO FINANCEIRO

RESUMO DA ETAPA 14
PowerPoint criado: Apresentador360_V2_Etapa14_Financeiro.pptx
Slides atualizados: 14 e 17
Total de vidas: 24
Valor atual: R$24.993
Valor reajustado: R$29.224
Cenários CC: R$12.356 | R$14.872 | R$15.140
Cenários SC: R$17.606 | R$20.373 | R$21.469
Quantidade de slides: 28
Relatório criado: Apresentador360_V2_Relatorio_Etapa14.txt

✅ ETAPA 14 CONCLUÍDA COM SUCESSO


In [43]:
# ============================================================
# APRESENTADOR 360 V2
# ETAPA 15
# PREENCHER SLIDES 15, 16, 18 E 19
# ANÁLISE FINANCEIRA CC E SC
# + GERAR POWERPOINT FINAL
# ============================================================

import os
import re

from openpyxl import load_workbook
from pptx import Presentation


print("=" * 70)
print("ETAPA 15 - PREENCHENDO ANÁLISES FINANCEIRAS")
print("=" * 70)


# ============================================================
# CONFIGURAÇÃO
# ============================================================

if "estado" not in globals():
    raise NameError(
        "O estado do projeto não existe. "
        "Execute as etapas anteriores."
    )


ARQUIVO_EXCEL_15 = estado.get(
    "arquivo_etapa_10",
    "Apresentador360_V2_Etapa10.xlsx"
)

ARQUIVO_PPT_ENTRADA_15 = estado.get(
    "arquivo_ppt_etapa_14",
    "Apresentador360_V2_Etapa14_Financeiro.pptx"
)

ARQUIVO_PPT_SAIDA_15 = (
    "Apresentador360_V2_FINAL.pptx"
)

ARQUIVO_RELATORIO_15 = (
    "Apresentador360_V2_Relatorio_Final.txt"
)


if not os.path.exists(ARQUIVO_EXCEL_15):
    raise FileNotFoundError(
        "Excel não encontrado: "
        + ARQUIVO_EXCEL_15
    )


if not os.path.exists(ARQUIVO_PPT_ENTRADA_15):
    raise FileNotFoundError(
        "PowerPoint não encontrado: "
        + ARQUIVO_PPT_ENTRADA_15
    )


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def limpar_texto_15(valor):

    if valor is None:
        return ""

    return " ".join(
        str(valor).replace("\n", " ").split()
    ).strip()


def numero_15(valor):

    try:
        return float(valor or 0)

    except (TypeError, ValueError):
        return 0.0


def eh_formula_15(valor):

    return (
        isinstance(valor, str)
        and valor.startswith("=")
    )


def eh_placeholder_15(valor):

    texto = limpar_texto_15(
        valor
    ).upper()

    return bool(
        re.fullmatch(
            r"PRODUTO\s+[0-9]+",
            texto
        )
    )


def formatar_numero_15(valor):

    numero = numero_15(valor)

    if abs(numero) < 0.005:
        return "-"

    return (
        f"{numero:,.2f}"
        .replace(",", "#")
        .replace(".", ",")
        .replace("#", ".")
    )


def formatar_inteiro_15(valor):

    numero = round(
        numero_15(valor)
    )

    if numero == 0:
        return "-"

    return str(numero)


def definir_texto_celula_15(
    celula,
    novo_texto
):
    """
    Atualiza texto da célula tentando preservar
    a formatação do primeiro trecho existente.
    """

    text_frame = celula.text_frame

    formato = {
        "nome": None,
        "tamanho": None,
        "negrito": None,
        "italico": None,
        "cor": None,
    }

    for paragrafo in text_frame.paragraphs:

        if not paragrafo.runs:
            continue

        run = paragrafo.runs[0]

        try:
            formato["nome"] = run.font.name
        except Exception:
            pass

        try:
            formato["tamanho"] = run.font.size
        except Exception:
            pass

        try:
            formato["negrito"] = run.font.bold
        except Exception:
            pass

        try:
            formato["italico"] = run.font.italic
        except Exception:
            pass

        try:
            formato["cor"] = run.font.color.rgb
        except Exception:
            pass

        break

    text_frame.clear()

    paragrafo = text_frame.paragraphs[0]

    novo_run = paragrafo.add_run()

    novo_run.text = str(
        novo_texto
    )

    if formato["nome"]:
        novo_run.font.name = formato["nome"]

    if formato["tamanho"]:
        novo_run.font.size = formato["tamanho"]

    if formato["negrito"] is not None:
        novo_run.font.bold = formato["negrito"]

    if formato["italico"] is not None:
        novo_run.font.italic = formato["italico"]

    if formato["cor"] is not None:

        try:
            novo_run.font.color.rgb = formato["cor"]
        except Exception:
            pass


def localizar_tabelas_slide_15(
    slide
):

    return [
        shape.table
        for shape in slide.shapes
        if getattr(
            shape,
            "has_table",
            False
        )
    ]


def localizar_linha_rotulo_15(
    tabela,
    rotulos
):

    rotulos_normalizados = [
        limpar_texto_15(
            rotulo
        ).upper()
        for rotulo in rotulos
    ]

    resultados = []

    for indice_linha, linha in enumerate(
        tabela.rows
    ):

        for indice_coluna, celula in enumerate(
            linha.cells
        ):

            texto = limpar_texto_15(
                celula.text
            ).upper()

            if texto in rotulos_normalizados:

                resultados.append({
                    "linha": indice_linha,
                    "coluna": indice_coluna,
                    "rotulo": texto,
                })

    return resultados


# ============================================================
# ABRIR EXCEL
# ============================================================

wb = load_workbook(
    ARQUIVO_EXCEL_15,
    data_only=False
)


ABAS_OBRIGATORIAS_15 = [
    "Planos",
    "Produtos Atuais",
    "Análise Financeira CC",
    "Análise Financeira SC",
]


for aba in ABAS_OBRIGATORIAS_15:

    if aba not in wb.sheetnames:

        wb.close()

        raise ValueError(
            "Aba não encontrada: "
            + aba
        )


ws_planos = wb["Planos"]

ws_produtos = wb[
    "Produtos Atuais"
]

ws_cc = wb[
    "Análise Financeira CC"
]

ws_sc = wb[
    "Análise Financeira SC"
]


# ============================================================
# PREPARAR VIDAS E PRODUTOS ATUAIS
# ============================================================

vidas_produtos_15 = []

produtos_atuais_15 = []

custos_atuais_15 = []


for indice_produto in range(10):

    linha_produto_atual = (
        2 + indice_produto
    )

    coluna_planos = (
        3 + indice_produto
    )

    plano = limpar_texto_15(
        ws_produtos.cell(
            linha_produto_atual,
            1
        ).value
    )

    vidas = numero_15(
        ws_produtos.cell(
            linha_produto_atual,
            2
        ).value
    )

    custo_atual = numero_15(
        ws_planos.cell(
            11,
            coluna_planos
        ).value
    )

    produtos_atuais_15.append(
        plano
    )

    vidas_produtos_15.append(
        vidas
    )

    custos_atuais_15.append(
        custo_atual
    )


# ============================================================
# PREPARAR FATORES DE REAJUSTE
# ============================================================

LINHAS_CUSTO_ANALISE_15 = [
    5,
    10,
    15,
    20,
    25,
    30,
    35,
    40,
    45,
    50,
]


fatores_reajuste_15 = []


for linha_custo in LINHAS_CUSTO_ANALISE_15:

    formula = limpar_texto_15(
        ws_cc.cell(
            linha_custo,
            3
        ).value
    )

    fator = 1.0

    referencia = re.search(
        r"\*([A-Z]+)(\d+)",
        formula
    )

    if referencia:

        coluna = referencia.group(1)

        linha = int(
            referencia.group(2)
        )

        fator_lido = numero_15(
            ws_cc[
                coluna + str(linha)
            ].value
        )

        if fator_lido > 0:
            fator = fator_lido

    fatores_reajuste_15.append(
        fator
    )


# ============================================================
# IDENTIFICAR CENÁRIOS NA ABA PLANOS
# ============================================================

linhas_players_15 = [
    4,
    5,
    6,
]


linhas_custo_cc_15 = [
    13,
    16,
    19,
]


linhas_custo_sc_15 = [
    14,
    17,
    20,
]


nomes_cenarios_15 = []

produtos_cenarios_15 = []


for linha_player in linhas_players_15:

    nome = limpar_texto_15(
        ws_planos.cell(
            linha_player,
            2
        ).value
    )

    nomes_cenarios_15.append(
        nome
    )

    produtos_player = []

    for coluna in range(
        3,
        13
    ):

        produtos_player.append(
            limpar_texto_15(
                ws_planos.cell(
                    linha_player,
                    coluna
                ).value
            )
        )

    produtos_cenarios_15.append(
        produtos_player
    )


# ============================================================
# MONTAR MATRIZ FINANCEIRA
# ============================================================

def montar_matriz_financeira_15(
    linhas_custos_cenarios
):

    registros = []

    for indice_produto in range(10):

        vidas = vidas_produtos_15[
            indice_produto
        ]

        produto_atual = produtos_atuais_15[
            indice_produto
        ]

        custo_atual = custos_atuais_15[
            indice_produto
        ]

        fator = fatores_reajuste_15[
            indice_produto
        ]

        custo_reajuste = (
            custo_atual * fator
        )

        opcoes = [
            {
                "player": limpar_texto_15(
                    ws_planos["B3"].value
                ),
                "produto": produto_atual,
                "vidas": vidas,
                "custo": custo_atual,
                "total": vidas * custo_atual,
            },
            {
                "player": "Reajuste",
                "produto": produto_atual,
                "vidas": vidas,
                "custo": custo_reajuste,
                "total": vidas * custo_reajuste,
            },
        ]

        for indice_cenario, linha_custo in enumerate(
            linhas_custos_cenarios
        ):

            produto_cenario = (
                produtos_cenarios_15[
                    indice_cenario
                ][
                    indice_produto
                ]
            )

            custo_cenario = numero_15(
                ws_planos.cell(
                    linha_custo,
                    3 + indice_produto
                ).value
            )

            opcoes.append({
                "player": nomes_cenarios_15[
                    indice_cenario
                ],
                "produto": produto_cenario,
                "vidas": vidas,
                "custo": custo_cenario,
                "total": vidas * custo_cenario,
            })

        registros.append({
            "indice": indice_produto + 1,
            "opcoes": opcoes,
        })

    return registros


matriz_cc_15 = montar_matriz_financeira_15(
    linhas_custo_cc_15
)

matriz_sc_15 = montar_matriz_financeira_15(
    linhas_custo_sc_15
)


wb.close()


# ============================================================
# ABRIR POWERPOINT DA ETAPA 14
# ============================================================

ppt = Presentation(
    ARQUIVO_PPT_ENTRADA_15
)


if len(ppt.slides) != 28:

    raise ValueError(
        "O PowerPoint deve possuir 28 slides."
    )


# ============================================================
# FUNÇÃO PARA PREENCHER TABELAS
# ============================================================

def preencher_slide_analise_15(
    slide,
    registros,
    indice_inicial
):
    """
    Preenche os blocos financeiros encontrados
    nas tabelas do slide.

    Cada bloco possui:
    Produto
    Número de Vidas
    Custo Médio
    Total
    """

    tabelas = localizar_tabelas_slide_15(
        slide
    )

    produto_atual = indice_inicial

    blocos_preenchidos = 0

    celulas_atualizadas = 0

    tabelas_processadas = 0


    for tabela in tabelas:

        ocorrencias_produto = (
            localizar_linha_rotulo_15(
                tabela,
                [
                    "Produto",
                ]
            )
        )

        if not ocorrencias_produto:
            continue

        tabelas_processadas += 1


        for ocorrencia in ocorrencias_produto:

            if produto_atual >= len(
                registros
            ):
                break

            linha_produto = ocorrencia[
                "linha"
            ]

            coluna_rotulo = ocorrencia[
                "coluna"
            ]

            registro = registros[
                produto_atual
            ]

            linhas_alvo = {
                "produto": linha_produto,
                "vidas": None,
                "custo": None,
                "total": None,
            }


            # Procurar os demais rótulos abaixo
            # ou próximos do Produto.
            for deslocamento in range(
                1,
                5
            ):

                linha_teste = (
                    linha_produto
                    + deslocamento
                )

                if linha_teste >= len(
                    tabela.rows
                ):
                    break

                texto_rotulo = (
                    limpar_texto_15(
                        tabela.cell(
                            linha_teste,
                            coluna_rotulo
                        ).text
                    ).upper()
                )

                if (
                    "NÚMERO DE VIDAS"
                    in texto_rotulo
                    or "NUMERO DE VIDAS"
                    in texto_rotulo
                    or texto_rotulo
                    == "VIDAS"
                ):
                    linhas_alvo[
                        "vidas"
                    ] = linha_teste

                elif (
                    "CUSTO MÉDIO"
                    in texto_rotulo
                    or "CUSTO MEDIO"
                    in texto_rotulo
                ):
                    linhas_alvo[
                        "custo"
                    ] = linha_teste

                elif texto_rotulo == "TOTAL":
                    linhas_alvo[
                        "total"
                    ] = linha_teste


            quantidade_colunas_dados = (
                len(tabela.columns)
                - coluna_rotulo
                - 1
            )

            quantidade_opcoes = min(
                quantidade_colunas_dados,
                len(registro["opcoes"])
            )


            for indice_opcao in range(
                quantidade_opcoes
            ):

                coluna_destino = (
                    coluna_rotulo
                    + 1
                    + indice_opcao
                )

                opcao = registro[
                    "opcoes"
                ][
                    indice_opcao
                ]


                definir_texto_celula_15(
                    tabela.cell(
                        linhas_alvo["produto"],
                        coluna_destino
                    ),
                    opcao["produto"]
                    if (
                        opcao["produto"]
                        and not eh_placeholder_15(
                            opcao["produto"]
                        )
                    )
                    else "-"
                )

                celulas_atualizadas += 1


                if linhas_alvo[
                    "vidas"
                ] is not None:

                    definir_texto_celula_15(
                        tabela.cell(
                            linhas_alvo["vidas"],
                            coluna_destino
                        ),
                        formatar_inteiro_15(
                            opcao["vidas"]
                        )
                    )

                    celulas_atualizadas += 1


                if linhas_alvo[
                    "custo"
                ] is not None:

                    definir_texto_celula_15(
                        tabela.cell(
                            linhas_alvo["custo"],
                            coluna_destino
                        ),
                        formatar_numero_15(
                            opcao["custo"]
                        )
                    )

                    celulas_atualizadas += 1


                if linhas_alvo[
                    "total"
                ] is not None:

                    definir_texto_celula_15(
                        tabela.cell(
                            linhas_alvo["total"],
                            coluna_destino
                        ),
                        formatar_numero_15(
                            opcao["total"]
                        )
                    )

                    celulas_atualizadas += 1


            produto_atual += 1

            blocos_preenchidos += 1


    return {
        "tabelas_encontradas": len(tabelas),
        "tabelas_processadas": tabelas_processadas,
        "blocos_preenchidos": blocos_preenchidos,
        "celulas_atualizadas": celulas_atualizadas,
        "proximo_produto": produto_atual,
    }


# ============================================================
# PREENCHER SLIDES 15 E 16 - CC
# ============================================================

resultado_slide_15 = preencher_slide_analise_15(
    slide=ppt.slides[14],
    registros=matriz_cc_15,
    indice_inicial=0,
)


resultado_slide_16 = preencher_slide_analise_15(
    slide=ppt.slides[15],
    registros=matriz_cc_15,
    indice_inicial=resultado_slide_15[
        "proximo_produto"
    ],
)


# ============================================================
# PREENCHER SLIDES 18 E 19 - SC
# ============================================================

resultado_slide_18 = preencher_slide_analise_15(
    slide=ppt.slides[17],
    registros=matriz_sc_15,
    indice_inicial=0,
)


resultado_slide_19 = preencher_slide_analise_15(
    slide=ppt.slides[18],
    registros=matriz_sc_15,
    indice_inicial=resultado_slide_18[
        "proximo_produto"
    ],
)


# ============================================================
# SALVAR POWERPOINT FINAL
# ============================================================

if os.path.exists(
    ARQUIVO_PPT_SAIDA_15
):

    os.remove(
        ARQUIVO_PPT_SAIDA_15
    )


ppt.save(
    ARQUIVO_PPT_SAIDA_15
)


# ============================================================
# VALIDAR ARQUIVO FINAL
# ============================================================

ppt_validacao_15 = Presentation(
    ARQUIVO_PPT_SAIDA_15
)


quantidade_slides_final_15 = len(
    ppt_validacao_15.slides
)


if quantidade_slides_final_15 != 28:

    raise RuntimeError(
        "A quantidade de slides foi alterada."
    )


resultados_slides_15 = {
    15: resultado_slide_15,
    16: resultado_slide_16,
    18: resultado_slide_18,
    19: resultado_slide_19,
}


total_tabelas_processadas_15 = sum(
    resultado[
        "tabelas_processadas"
    ]
    for resultado in (
        resultados_slides_15.values()
    )
)


total_blocos_preenchidos_15 = sum(
    resultado[
        "blocos_preenchidos"
    ]
    for resultado in (
        resultados_slides_15.values()
    )
)


total_celulas_atualizadas_15 = sum(
    resultado[
        "celulas_atualizadas"
    ]
    for resultado in (
        resultados_slides_15.values()
    )
)


# ============================================================
# CRIAR RELATÓRIO FINAL
# ============================================================

with open(
    ARQUIVO_RELATORIO_15,
    "w",
    encoding="utf-8"
) as arquivo:

    arquivo.write(
        "=" * 70 + "\n"
    )

    arquivo.write(
        "APRESENTADOR 360 V2 - ETAPA 15\n"
    )

    arquivo.write(
        "CONSOLIDAÇÃO FINAL DO POWERPOINT\n"
    )

    arquivo.write(
        "=" * 70 + "\n\n"
    )

    arquivo.write(
        "Excel utilizado: "
        + ARQUIVO_EXCEL_15
        + "\n"
    )

    arquivo.write(
        "PowerPoint de entrada: "
        + ARQUIVO_PPT_ENTRADA_15
        + "\n"
    )

    arquivo.write(
        "PowerPoint final: "
        + ARQUIVO_PPT_SAIDA_15
        + "\n\n"
    )

    arquivo.write(
        "Slide 13: Painel Demográfico\n"
    )

    arquivo.write(
        "Slide 14: Investimento Financeiro CC\n"
    )

    arquivo.write(
        "Slide 15: Análise Financeira CC\n"
    )

    arquivo.write(
        "Slide 16: Análise Financeira CC\n"
    )

    arquivo.write(
        "Slide 17: Investimento Financeiro SC\n"
    )

    arquivo.write(
        "Slide 18: Análise Financeira SC\n"
    )

    arquivo.write(
        "Slide 19: Análise Financeira SC\n"
    )

    arquivo.write(
        "Slides 20 a 28: preservados para edição manual\n\n"
    )

    for numero_slide, resultado in (
        resultados_slides_15.items()
    ):

        arquivo.write(
            "SLIDE "
            + str(numero_slide)
            + "\n"
        )

        arquivo.write(
            "Tabelas encontradas: "
            + str(
                resultado[
                    "tabelas_encontradas"
                ]
            )
            + "\n"
        )

        arquivo.write(
            "Tabelas processadas: "
            + str(
                resultado[
                    "tabelas_processadas"
                ]
            )
            + "\n"
        )

        arquivo.write(
            "Blocos preenchidos: "
            + str(
                resultado[
                    "blocos_preenchidos"
                ]
            )
            + "\n"
        )

        arquivo.write(
            "Células atualizadas: "
            + str(
                resultado[
                    "celulas_atualizadas"
                ]
            )
            + "\n\n"
        )

    arquivo.write(
        "Total de tabelas processadas: "
        + str(
            total_tabelas_processadas_15
        )
        + "\n"
    )

    arquivo.write(
        "Total de blocos preenchidos: "
        + str(
            total_blocos_preenchidos_15
        )
        + "\n"
    )

    arquivo.write(
        "Total de células atualizadas: "
        + str(
            total_celulas_atualizadas_15
        )
        + "\n"
    )

    arquivo.write(
        "Quantidade final de slides: "
        + str(
            quantidade_slides_final_15
        )
        + "\n"
    )


# ============================================================
# ATUALIZAR ESTADO
# ============================================================

estado["arquivo_ppt_final"] = (
    ARQUIVO_PPT_SAIDA_15
)

estado["relatorio_final"] = (
    ARQUIVO_RELATORIO_15
)

estado["resultado_etapa_15"] = {
    "slides": resultados_slides_15,
    "tabelas_processadas": (
        total_tabelas_processadas_15
    ),
    "blocos_preenchidos": (
        total_blocos_preenchidos_15
    ),
    "celulas_atualizadas": (
        total_celulas_atualizadas_15
    ),
    "quantidade_slides": (
        quantidade_slides_final_15
    ),
}


# ============================================================
# RESUMO
# ============================================================

print("\n" + "=" * 70)
print("RESUMO DA ETAPA 15")
print("=" * 70)

print(
    "PowerPoint final:",
    ARQUIVO_PPT_SAIDA_15
)

print(
    "Slides financeiros preenchidos:",
    "14, 15, 16, 17, 18 e 19"
)

print(
    "Tabelas processadas:",
    total_tabelas_processadas_15
)

print(
    "Blocos preenchidos:",
    total_blocos_preenchidos_15
)

print(
    "Células atualizadas:",
    total_celulas_atualizadas_15
)

print(
    "Slides 20 a 28 preservados:",
    "SIM"
)

print(
    "Quantidade de slides:",
    quantidade_slides_final_15
)

print(
    "Relatório final:",
    ARQUIVO_RELATORIO_15
)

print(
    "\n✅ ETAPA 15 CONCLUÍDA COM SUCESSO"
)

print("=" * 70)

ETAPA 15 - PREENCHENDO ANÁLISES FINANCEIRAS

RESUMO DA ETAPA 15
PowerPoint final: Apresentador360_V2_FINAL.pptx
Slides financeiros preenchidos: 14, 15, 16, 17, 18 e 19
Tabelas processadas: 4
Blocos preenchidos: 10
Células atualizadas: 200
Slides 20 a 28 preservados: SIM
Quantidade de slides: 28
Relatório final: Apresentador360_V2_Relatorio_Final.txt

✅ ETAPA 15 CONCLUÍDA COM SUCESSO


In [55]:
# ============================================================
# APRESENTADOR 360 V2
# AJUSTE FINAL DO SLIDE 13
# RESTAURAR LAYOUT DO BLOCO GÊNERO
# ============================================================

import os
import re
from copy import deepcopy

from pptx import Presentation


print("=" * 70)
print("AJUSTE FINAL - CORRIGINDO O BLOCO GÊNERO DO SLIDE 13")
print("=" * 70)


# ============================================================
# ARQUIVOS
# ============================================================

ARQUIVO_PPT_ORIGINAL = (
    "Mezzo_Apresentação Saúde_2026_07_v1.pptx"
)

ARQUIVO_PPT_FINAL = estado.get(
    "arquivo_ppt_final",
    "Apresentador360_V2_FINAL.pptx"
)

ARQUIVO_PPT_CORRIGIDO = (
    "Apresentador360_V2_FINAL_CORRIGIDO.pptx"
)


for arquivo_obrigatorio in [
    ARQUIVO_PPT_ORIGINAL,
    ARQUIVO_PPT_FINAL,
]:

    if not os.path.exists(arquivo_obrigatorio):

        raise FileNotFoundError(
            "Arquivo não encontrado: "
            + arquivo_obrigatorio
        )


# ============================================================
# DADOS DEMOGRÁFICOS
# ============================================================

elegibilidade = estado["elegibilidade"]

total_vidas = int(
    elegibilidade.get(
        "total_vidas",
        0
    )
    or 0
)

feminino = int(
    elegibilidade.get(
        "feminino",
        0
    )
    or 0
)

masculino = int(
    elegibilidade.get(
        "masculino",
        0
    )
    or 0
)


percentual_feminino = round(
    feminino / total_vidas * 100
) if total_vidas else 0

percentual_masculino = round(
    masculino / total_vidas * 100
) if total_vidas else 0


# ========================

AJUSTE FINAL - CORRIGINDO O BLOCO GÊNERO DO SLIDE 13


In [51]:
import os

for arquivo in sorted(os.listdir()):
    if arquivo.lower().endswith(".pptx"):
        print(arquivo)

Apresentador360_V2_Etapa13_Painel_Demografico.pptx
Apresentador360_V2_Etapa14_Financeiro.pptx
Apresentador360_V2_FINAL.pptx
Mezzo_Apresentação Saúde_2026_07_v1.pptx


In [56]:
from google.colab import files

files.download(
    "Apresentador360_V2_FINAL.pptx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>